# 08 — Characterising the pre-Haiyan nighttime lights baseline

How often can Samar–Leyte be observed, how regularly do near-zero nights recur, and how much does the pre-event reference depend on settlement mask and viewing geometry? G1 at 25% spatial completeness is the primary series. G2–G8 show how progressively broader settlement masks change the signal.

Each chart has a named Plotly figure and a separate display and PNG export cell. Edit its traces, axes and layout locally. Run the preparation cells once when starting a fresh kernel.

## 1. Inputs and analysis choices

| Input or setting | Role |
|---|---|
| Processed baseline and event-period VNP46A2 Zarr stores | DNB-BRDF and mandatory quality flag (MQF) |
| GHSL SMOD 2015 and municipal boundaries | Fixed raster support for G1 (GHSL class 30) |
| VNP46A1 stores, if present | Same-pixel, same-date sensor zenith |
| G1 with ≥25% daily SC | Primary regional configuration informed by revised RQ1 |
| MQF ∈ {0, 1} | HQ retrievals throughout the notebook |
| Last 60 pre-event days | Primary reference window, compared with 30, 90 and 180 days |

SC is the percentage of all fixed pixels in a settlement mask with finite eligible radiance on that date. An available scene with zero eligible pixels has 0% SC. An unavailable scene has undefined SC. No positive-radiance or baseline-completeness condition changes the fixed denominator.

The primary statistic is the daily spatial median. The daily P95-capped mean and temporally clipped mean remain sensitivity comparisons. PNG exports use Plotly’s Kaleido renderer. CSV outputs retain their existing paths.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
import geopandas as gpd
from rasterio.enums import Resampling
from rasterio.features import rasterize
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
from IPython.display import display

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (PROJECT_DIR / "datasets").exists(): PROJECT_DIR = Path('/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery')
DATA_DIR = PROJECT_DIR / "datasets"
PROCESSED_DIR = DATA_DIR / "VNP46/processed"
BASELINE_STORE = PROCESSED_DIR / "baseline/Haiyan_VNP46A2_baseline.zarr"
EXISTING_STORE = PROCESSED_DIR / "Haiyan_VNP46A2.zarr"
if not BASELINE_STORE.exists() or not EXISTING_STORE.exists():
    raise FileNotFoundError(f"Expected both processed VNP46A2 stores under {PROCESSED_DIR}. Set PROJECT_DIR above.")
GHSL_PATH = next(DATA_DIR.glob("**/GHSL_SMOD_E2015.tif"), None)
if GHSL_PATH is None:
    raise FileNotFoundError(f"GHSL_SMOD_E2015.tif is missing under {DATA_DIR}.")
MUNICIPALITIES_PATH = DATA_DIR / "boundaries/MuniCities/MuniCities.shp"
OUTPUT_DIR = PROJECT_DIR / "output/baseline_cyclicity"
FIGURE_DIR, TABLE_DIR = OUTPUT_DIR / "figures", OUTPUT_DIR / "tables"
for path in (FIGURE_DIR,TABLE_DIR): path.mkdir(parents=True,exist_ok=True)
EVENT_DATE = pd.Timestamp("2013-11-08")
BASELINE_DAYS, AGGREGATION_DAYS = 60, 4
BASELINE_START = EVENT_DATE-pd.Timedelta(days=BASELINE_DAYS)
PRE_EVENT_END = EVENT_DATE-pd.Timedelta(days=1)
CHARACTERIZATION_START = pd.Timestamp("2012-01-19")

G1_CLASSES = (30,)
HQ_MQF = (0, 1)
SPATIAL_COMPLETENESS_PCT = 25.0
RQ_CLIP_PERCENTILE = 95.0
MIN_REFERENCE_DAYS = 10
LOW_NTL = .05
KNOWN_FILL_VALUES = (-9999., -999., -32768., 6553.5, 65535.)
DNB_BAND, MQF_BAND = "DNB_BRDF_Corrected_NTL", "Mandatory_Quality_Flag"
PLOT_FONT, PLOT_TEXT_COLOR, PLOT_GRID_COLOR = "Arial", "#243B5A", "#E8EDF3"
EVENT_COLOR, BASELINE_COLOR = "#0057FF", "#6C7882"
PRIMARY_COLOR, EXPANDED_COLOR, AGGREGATE_COLOR = "#0072B2", "#CA6400", "#008A61"
NTL_UNITS = "nW cm⁻² sr⁻¹"
PNG_SCALE = 2

/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def style_figure(fig,title,width=1280,height=720):
    fig.update_layout(template="plotly_white",paper_bgcolor="rgba(0,0,0,0)",plot_bgcolor="white",
        width=width,height=height,font=dict(family=PLOT_FONT,size=16,color=PLOT_TEXT_COLOR),
        title=dict(text=title,x=.03,xanchor="left",font=dict(size=23)),
        margin=dict(l=100,r=115,t=110,b=115),
        legend=dict(orientation="h",x=0,y=-.09,xanchor="left",yanchor="top"))
    fig.update_xaxes(zeroline=False,gridcolor=PLOT_GRID_COLOR)
    fig.update_yaxes(zeroline=False,gridcolor=PLOT_GRID_COLOR)
    return fig

def add_haiyan_marker(fig,row=None,col=None):
    fig.add_vline(x=EVENT_DATE.to_pydatetime(),line_dash="dash",line_color=EVENT_COLOR,line_width=2,row=row,col=col)


def sc_heatmap(fig, coverage, row, col=1, bar_y=.14):
    fig.add_trace(go.Heatmap(
        x=coverage.index, y=coverage.columns,
        z=np.where(coverage.isna().T, 1., np.nan),
        colorscale=[[0, "#CFD5DC"], [1, "#CFD5DC"]], showscale=False,
        hoverongaps=False, hovertemplate="%{x|%d %b %Y} · %{y}<br>SC unavailable<extra></extra>",
    ), row=row, col=col)
    fig.add_trace(go.Heatmap(
        x=coverage.index, y=coverage.columns, z=coverage.T.to_numpy(), zmin=0, zmax=100,
        colorscale="Greens", zsmooth=False, hoverongaps=False,
        colorbar=dict(title="SC (%)", tickvals=[0, 50, 100], thickness=14, len=.22, y=bar_y),
        hovertemplate="%{x|%d %b %Y} · %{y}<br>SC %{z:.1f}%<extra></extra>",
    ), row=row, col=col)

def mark_reference(fig, rows):
    for row in rows:
        fig.add_vrect(x0=BASELINE_START, x1=EVENT_DATE, fillcolor=EVENT_COLOR,
                      opacity=.07, line_width=0, layer="below", row=row, col=1)
        add_haiyan_marker(fig, row=row, col=1)
    fig.add_annotation(x=EVENT_DATE, y=0.95, xref="x", yref="paper",
                       text="Haiyan", showarrow=False, xanchor="right",
                       font=dict(color=EVENT_COLOR, size=13))

def median(values, axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmedian(values, axis=axis)

## 2. Prepare daily radiance and fixed settlement masks

In [3]:
early = xr.open_zarr(BASELINE_STORE,chunks={},consolidated=False,mask_and_scale=False)
existing = xr.open_zarr(EXISTING_STORE,chunks={},consolidated=False,mask_and_scale=False)
early = early.sel(date=slice(CHARACTERIZATION_START, PRE_EVENT_END))
existing = existing.sel(date=slice(CHARACTERIZATION_START, PRE_EVENT_END))
if not np.all(early.processed.values == 1):
    raise RuntimeError("Complete the baseline A2 store for the pre-Haiyan dates.")
for axis in ("x","y"):
    np.testing.assert_allclose(early[axis].values,existing[axis].values,rtol=0,atol=1e-7)
early=early.assign_coords(x=existing.x,y=existing.y)
early["spatial_ref"]=existing["spatial_ref"]
complete=existing.processed.values==1
if len(pd.DatetimeIndex(existing.date.values[~complete]).difference(early.date.values)):
    raise RuntimeError("Existing event-period processing is unfinished.")
existing=existing.isel(date=np.flatnonzero(complete))
new_dates=pd.DatetimeIndex(early.date.values).difference(existing.date.values)
a2=xr.concat([early.sel(date=new_dates),existing],dim="date",data_vars="minimal",coords="minimal",compat="equals",join="exact").sortby("date")
a2=a2.rio.write_crs(existing.attrs["crs_wkt"])
template=a2[DNB_BAND].isel(date=0,drop=True)
ghsl=rxr.open_rasterio(GHSL_PATH,masked=True).squeeze("band",drop=True)
ghsl=ghsl.rio.reproject_match(template,resampling=Resampling.nearest).values
municipalities=gpd.read_file(MUNICIPALITIES_PATH)
region=municipalities[municipalities.PROVINCE.str.strip().str.title().isin(
    ["Samar","Eastern Samar","Northern Samar","Leyte","Southern Leyte"])].to_crs(a2.rio.crs).copy()
region["zone_id"]=np.arange(1,len(region)+1)
zone=rasterize([(g,int(i)) for g,i in zip(region.geometry,region.zone_id)],out_shape=ghsl.shape,
               transform=template.rio.transform(recalc=True),fill=0)


land = (zone > 0) & np.isin(ghsl, G1_CLASSES)
land_pixels = np.flatnonzero(land)
n_g1 = len(land_pixels)
if not n_g1:
    raise ValueError("No GHSL 30 pixels in the fixed Samar–Leyte footprint.")

daily_calendar = pd.date_range(CHARACTERIZATION_START, PRE_EVENT_END, name="date")
dates = pd.DatetimeIndex(a2.date.values)
if dates.empty:
    raise ValueError("No processed A2 scenes in the pre-Haiyan period.")
if not dates.is_unique:
    raise ValueError("The processed A2 stores contain duplicate daily dates.")
pixel_values = np.full((len(daily_calendar), n_g1), np.nan, dtype="float32")

records = []
for start in tqdm(range(0, len(dates), 8), desc="Read pre-Haiyan G1 HQ retrievals"):
    batch = a2[[DNB_BAND, MQF_BAND]].isel(date=slice(start, start + 8))
    batch = batch.transpose("date", "y", "x", missing_dims="ignore").compute()
    shape = (batch.sizes["date"], -1)
    radiance = batch[DNB_BAND].values.reshape(shape)[:, land_pixels].astype("float32")
    quality = batch[MQF_BAND].values.reshape(shape)[:, land_pixels]
    valid = (np.isfinite(radiance) & (radiance >= 0)
             & ~np.isin(radiance, KNOWN_FILL_VALUES) & np.isin(quality, HQ_MQF))
    radiance[~valid] = np.nan

    for date, day, keep, flags in zip(pd.DatetimeIndex(batch.date.values), radiance, valid, quality):
        pixel_values[daily_calendar.get_loc(date)] = day
        values = day[keep]
        sc = 100 * values.size / n_g1
        if values.size:
            cap = np.percentile(values, RQ_CLIP_PERCENTILE)
            stats = (values.mean(), np.clip(values, 0, cap).mean(), np.median(values), cap,
                     100 * np.mean(values == 0),
                     100 * np.isclose(values, .010, atol=1e-6, rtol=0).mean(),
                     100 * np.mean(flags[keep] == 1))
        else:
            stats = (np.nan,) * 7
        records.append((date, values.size, sc, *stats))

primary = pd.DataFrame(records, columns=[
    "date", "n", "SC", "mean", "mean_capped", "median", "P95",
    "zero_pct", "at_001_pct", "mqf1_pct",
]).set_index("date").reindex(daily_calendar)
primary.assign(mask="G1", retrievals="HQ", qualified=primary.SC.ge(SPATIAL_COMPLETENESS_PCT)).to_csv(
    TABLE_DIR / "pre_haiyan_daily_diagnostics.csv"
)

Read pre-Haiyan G1 HQ retrievals: 100%|██████████| 82/82 [00:01<00:00, 51.54it/s]


In [4]:
profiles = {
    "G1": ((30,),        "#FF0000"),
    "G2": ((23, 30),     "#732600"),
    "G3": ((22, 23, 30), "#A87000"),
}

plot_end = pd.Timestamp("2014-01-31")
plot_dates = pd.date_range(CHARACTERIZATION_START, plot_end, name="date")
gate = tau = SPATIAL_COMPLETENESS_PCT

# Select pixels from the full regional grid. Notebook 08's land_pixels is G1-only.
plot_groups = ("G1", "G2", "G3")
class_order = (30, 23, 22, 21, 13, 12, 11, 10)
for i, color in enumerate(
    ("#FFFF00", "#375623", "#ABCD66", "#CDF57A", "#7AB6F5"), start=4
):
    profiles[f"G{i}"] = (class_order[:i], color)

groups = [f"G{i}" for i in range(1, 9)]
profile_pixels = {
    g: np.flatnonzero((zone > 0) & np.isin(ghsl, profiles[g][0]))
    for g in groups
}
for g, idx in profile_pixels.items():
    if idx.size == 0:
        raise ValueError(f"No regional pixels found for {g}.")

# Read the combined footprint once per batch. Keep each mask's denominator fixed.
plot_pixels = np.unique(np.concatenate(list(profile_pixels.values())))
mask_cols = {g: np.searchsorted(plot_pixels, idx) for g, idx in profile_pixels.items()}

daily_by_mask = {
    g: pd.DataFrame(np.nan, index=plot_dates, columns=["n", "SC", "median"])
    for g in groups
}

def summarise_profile_dates(source):
    for start in tqdm(range(0, source.sizes["date"], 8), desc="G1–G8 daily HQ medians"):
        batch = (
            source[[DNB_BAND, MQF_BAND]]
            .isel(date=slice(start, start + 8))
            .transpose("date", "y", "x")
            .compute()
        )
        batch_dates = pd.DatetimeIndex(batch.date.values)
        shape = (len(batch_dates), -1)
        raw = batch[DNB_BAND].values.reshape(shape)[:, plot_pixels].astype("float32")
        mqf = batch[MQF_BAND].values.reshape(shape)[:, plot_pixels]

        valid = (
            np.isfinite(raw) & (raw >= 0)
            & ~np.isin(raw, KNOWN_FILL_VALUES)
            & np.isin(mqf, [0, 1])
        )
        raw[~valid] = np.nan

        for g, cols in mask_cols.items():
            day_values = raw[:, cols]
            counts = np.isfinite(day_values).sum(axis=1)
            medians = np.full(len(batch_dates), np.nan)

            for j in np.flatnonzero(counts):
                values = day_values[j, np.isfinite(day_values[j])]
                cap = np.percentile(values, RQ_CLIP_PERCENTILE)
                medians[j] = np.median(np.clip(values, 0, cap))

            daily_by_mask[g].loc[batch_dates, ["n", "SC", "median"]] = np.column_stack([
                counts, 100 * counts / len(cols), medians,
            ])

# Use a2 first. Some versions of Notebook 08 slice it at PRE_EVENT_END.
plot_a2 = a2.sel(date=slice(plot_dates[0], plot_end))
summarise_profile_dates(plot_a2)

# Recover additional processed dates through January 2014 from the event store.
if plot_a2.sizes["date"] == 0 or pd.Timestamp(plot_a2.date.values[-1]) < plot_end:
    with xr.open_zarr(EXISTING_STORE, chunks={}, consolidated=False,
                      mask_and_scale=False) as event_store:
        for axis in ("x", "y"):
            np.testing.assert_allclose(event_store[axis], a2[axis], rtol=0, atol=1e-7)

        event_part = event_store.sel(date=slice(plot_dates[0], plot_end))
        processed_dates = pd.DatetimeIndex(
            event_part.date.values[event_part["processed"].values == 1]
        )
        extra_dates = processed_dates.difference(plot_a2.date.values)
        if len(extra_dates):
            summarise_profile_dates(event_part.sel(date=extra_dates))

plot_daily = daily_by_mask["G1"]

# Check support before plotting. A missing scene has NaN SC.
# A processed scene with zero HQ pixels has SC = 0 and median = NaN.
display(pd.DataFrame({
    g: {
        "Fixed pixels": len(profile_pixels[g]),
        "Days with HQ NTL": frame["median"].count(),
        "Days at ≥25% SC": (frame["median"].notna() & frame["SC"].ge(gate)).sum(),
        "Last HQ date": frame["median"].last_valid_index(),
    }
    for g, frame in daily_by_mask.items()
}).T)

G1–G8 daily HQ medians: 100%|██████████| 11/11 [00:00<00:00, 19.56it/s]


,Fixed pixels,Days with HQ NTL,Days at ≥25% SC,Last HQ date
G1,503,453,327,2014-01-30 00:00:00
G2,1932,547,339,2014-01-30 00:00:00
G3,3426,567,337,2014-01-30 00:00:00
G4,13973,577,334,2014-01-30 00:00:00
G5,17803,588,340,2014-01-30 00:00:00
G6,33731,594,344,2014-01-30 00:00:00
G7,96173,597,357,2014-01-30 00:00:00
G8,96686,598,357,2014-01-30 00:00:00


## 3. Daily radiance and observability

The green strips show daily SC above each radiance profile. The step line takes the median of at least two qualified daily medians in a centred four-calendar-day window, split at landfall.

In [5]:
gate = SPATIAL_COMPLETENESS_PCT
percentile = RQ_CLIP_PERCENTILE
below_gate_label = f"<{gate:g}% SC"
outlier_label = f">{percentile:g}th percentile"
below_gate_color = "#9EA7B0"

fig_profiles = make_subplots(
    rows=6, cols=1, shared_xaxes=True,
    row_heights=[.055, .278, .055, .278, .055, .279],
    vertical_spacing=.025,
)
outlier_legend_shown = False

for panel, label in enumerate(plot_groups):
    color = profiles[label][1]
    sc_row = 2 * panel + 1
    ntl_row = sc_row + 1

    frame = daily_by_mask[label]
    daily = frame["median"]
    qualified = daily.where(frame["SC"].ge(gate))
    below_gate = daily.where(frame["SC"].lt(gate))

    # Split at Haiyan before computing the centred four-day median.
    phases = (
        qualified.loc[qualified.index < EVENT_DATE],
        qualified.loc[qualified.index >= EVENT_DATE],
    )
    windows = [
        part.rolling(AGGREGATION_DAYS, center=True, min_periods=2)
        for part in phases
    ]
    four_day = [window.median() for window in windows]

    axis_top = daily.quantile(percentile / 100)
    has_cap = np.isfinite(axis_top) and axis_top > 0
    above_axis = (
        qualified.gt(axis_top) if has_cap
        else pd.Series(False, index=daily.index)
    )
    below_gate_plot = below_gate.clip(upper=axis_top) if has_cap else below_gate

    # Spatial completeness strip.
    sc_heatmap(
        fig_profiles, frame[["SC"]].rename(columns={"SC": label}),
        row=sc_row, bar_y=.94,
    )
    fig_profiles.data[-1].update(
        colorscale=[[0, "#F7FCF5"], [.5, "#74C476"], [1, "#006D2C"]],
        showscale=(panel == 0),
        colorbar=dict(title="SC (%)", len=.1, x = 1.01, y=.99, thickness=15, tickvals=[0, 50, 100]),
    )
    fig_profiles.update_xaxes(showgrid=False, row=sc_row, col=1)
    fig_profiles.update_yaxes(showgrid=False, row=sc_row, col=1)

    # Daily observations and their unmodified values in hover.
    daily_hover = frame[["median", "SC", "n"]]
    for values, name, marker, legend_rank, show_legend in [
        (
            qualified.mask(above_axis), f"{label} daily",
            dict(color=color, size=4, opacity=.30), 2 * panel, True,
        ),
        (
            below_gate_plot, below_gate_label,
            dict(color=below_gate_color, size=5, symbol="x", opacity=.65),
            10, panel == 0,
        ),
    ]:
        fig_profiles.add_trace(go.Scatter(
            x=frame.index, y=values, mode="markers",
            name=name, legendgroup=name, legendrank=legend_rank,
            showlegend=show_legend, cliponaxis=False,
            marker=marker, customdata=daily_hover,
            hovertemplate=(
                "%{x|%d %b %Y}<br>Median %{customdata[0]:.3f}"
                "<br>SC %{customdata[1]:.1f}% · %{customdata[2]:.0f} pixels"
                f"<extra>{label}</extra>"
            ),
        ), row=ntl_row, col=1)

    # Four-day median, with one legend entry per mask.
    line_name = f"{label} 4D median"
    for phase, (smooth, window) in enumerate(zip(four_day, windows)):
        fig_profiles.add_trace(go.Scatter(
            x=smooth.index, y=smooth, mode="lines",
            name=line_name, legendgroup=line_name, legendrank=2 * panel + 1,
            showlegend=(phase == 0),
            line=dict(color=color, width=1.6, shape="hv"),
            connectgaps=False, customdata=window.count(),
            hovertemplate=(
                "%{x|%d %b %Y}4D median %{y:.3f}"
                "%{customdata:.0f} qualified days"
                f"<extra>{label}</extra>"
            ),
        ), row=ntl_row, col=1)

    # Qualified daily medians above the panel ceiling.
    outliers = frame.loc[above_axis]
    if not outliers.empty:
        fig_profiles.add_trace(go.Scatter(
            x=outliers.index, y=np.full(len(outliers), axis_top),
            mode="markers", name=outlier_label,
            legendgroup="above_percentile", legendrank=11,
            showlegend=not outlier_legend_shown, cliponaxis=False,
            marker=dict(color=color, size=6, symbol="x", opacity=.75),
            customdata=outliers[["median", "SC"]],
            hovertemplate=(
                "%{x|%d %b %Y}Actual median %{customdata[0]:.3f}"
                "SC %{customdata[1]:.1f}%"
                f"<extra>{label} · {outlier_label}</extra>"
            ),
        ), row=ntl_row, col=1)
        outlier_legend_shown = True

    fig_profiles.update_yaxes(title_text=NTL_UNITS, rangemode="tozero", row=ntl_row, col=1)
    if has_cap:
        fig_profiles.update_yaxes(range=[0, axis_top], row=ntl_row, col=1)

    for row in (sc_row, ntl_row):
        add_haiyan_marker(fig_profiles, row=row, col=1)

style_figure(fig_profiles, "", 1420, 860)
fig_profiles.update_layout(margin=dict(l=86, r=95, t=38, b=80))
fig_profiles.add_annotation(
    x=EVENT_DATE, y=.95, xref="x", yref="paper",
    text="Haiyan", showarrow=False, xanchor="left",
    font=dict(color=EVENT_COLOR, size=13),
)
fig_profiles.update_xaxes(
    range=[CHARACTERIZATION_START, plot_end + pd.Timedelta(days=1)],
    dtick="M3", tickformat="%b %Y", row=6, col=1,
)

In [6]:
fig_profiles.show()
fig_profiles.write_image(FIGURE_DIR / ("01_g1_g2_g3_ntl_and_sc_through_jan_2014" + ".png"), scale=PNG_SCALE)

### Discussion

The urban core is brighter than the broader masks. Adding GHSL 23 introduces repeated troughs in G2, and several coincide with G3 troughs. The upper-axis crosses retain unusually bright daily observations while keeping the four-day profile readable. A break in the line means insufficient observations in the window.

### Monthly G1 qualification

In [7]:
# 02. Monthly G1 qualification.
gate = SPATIAL_COMPLETENESS_PCT
status = pd.Series("Below coverage gate", index=plot_daily.index, name="Status")
status.loc[plot_daily["SC"].isna()] = "No A2 scene"
status.loc[plot_daily["SC"].ge(gate) & plot_daily["median"].notna()] = "Qualified HQ date"

order = ["Qualified HQ date", "Below coverage gate", "No A2 scene"]
labels = [f"SC ≥{gate:g}%", f"SC <{gate:g}%", "No scene"]
colors = ["#FF0000", "#939DA8", "#E5E9EE"]

monthly_counts = pd.crosstab(plot_daily.index.to_period("M"), status).reindex(
    columns=order, fill_value=0,
)
month_days = monthly_counts.sum(axis=1)
monthly_pct = monthly_counts.div(month_days, axis=0).mul(100)

month_start = monthly_pct.index.to_timestamp()
month_end = (monthly_pct.index + 1).to_timestamp()
month_center = month_start + (month_end - month_start) / 2
bar_width = (month_end - month_start).total_seconds() * 1000 * 0.85

In [8]:
colors = ["#238B45", "#9EA7B0", "#000000"]

fig_monthly = go.Figure()
for column, label, color in zip(order, labels, colors):
    fig_monthly.add_trace(go.Bar(
        x=month_center,
        y=monthly_pct[column],
        width=bar_width,
        name=label,
        marker_color=color,
        customdata=np.column_stack([monthly_counts[column], month_days]),
        hovertemplate=(
            "%{x|%b %Y}<br>%{y:.1f}% · %{customdata[0]}/%{customdata[1]} days"
            "<extra>%{fullData.name}</extra>"
        ),
    ))

# Put the qualified-day count just above each green segment.
qualified_column = "Qualified HQ date"
for x, share, count in zip(
    month_center,
    monthly_pct[qualified_column],
    monthly_counts[qualified_column],
):
    fig_monthly.add_annotation(
        x=x,
        y=share,
        text=str(int(count)),
        showarrow=False,
        yanchor="bottom",
        yshift=2,
        font=dict(size=20, color=colors[0]),
        borderpad=1,
    )

style_figure(fig_monthly, "", 1420, 355)
fig_monthly.update_layout(
    barmode="stack",
    margin=dict(l=75, r=30, t=15, b=95),
    legend=dict(
        orientation="h",
        x=0.5, xanchor="center",
        y=-0.22, yanchor="top",
    ),
)
fig_monthly.update_xaxes(
    type="date",
    range=[month_start[0], month_end[-1]],
    tickvals=month_center,
    ticktext=[
        date.strftime("%b<br>%Y") if date.month == 1 else date.strftime("%b")
        for date in month_start
    ],
    showgrid=False,
)
fig_monthly.update_yaxes(
    title_text="Calendar days",
    range=[0, 110],
    tickvals=[0, 25, 50, 75, 100],
    ticksuffix="%",
)
fig_monthly.show()

In [9]:
fig_monthly.show()
fig_monthly.write_image(FIGURE_DIR / ("02_monthly_qualification_through_jan_2014" + ".png"), scale=PNG_SCALE)

In [10]:
monthly_pct.to_csv(TABLE_DIR / "g1_monthly_qualification_pct_through_jan_2014.csv")

### Discussion

The monthly denominator includes every calendar day within the plotted interval, including the partial first month. The stacked bars distinguish days with a scene but insufficient HQ coverage from days without a processed scene. A month with few qualified days contributes correspondingly little evidence about typical radiance.

## 4. Near-zero dates across settlement masks

Diamonds identify daily spatial medians at or below 0.05. The scatter panels compare those observations with their coverage before Haiyan.

In [11]:
fig_dips = make_subplots(
    rows=len(groups),
    cols=2,
    shared_xaxes="columns",
    column_widths=[0.72, 0.28],
    horizontal_spacing=0.07,
    vertical_spacing=0.025,
)

# Give the scatter panels a shared log scale and a separate position for zero.
pre_values = pd.concat([
    daily_by_mask[g].reindex(plot_dates).loc[:PRE_EVENT_END, "median"]
    for g in groups
]).dropna()

positive = pre_values[pre_values.gt(0)]
log_min = min(0.01, positive.min()) if not positive.empty else 0.01
log_top = max(1.0, positive.max()) if not positive.empty else 1.0
zero_pos = 10 ** (np.floor(np.log10(log_min)) - 1)
has_zero = pre_values.eq(0).any()
log_bottom = zero_pos if has_zero else log_min

# Keep the labelled ticks sparse enough to read across eight rows.
log_ticks = [value for value in (0.01, 0.1, 1, 10) if log_min <= value <= log_top]
tickvals = ([zero_pos] if has_zero else []) + log_ticks
ticktext = (["0"] if has_zero else []) + [f"{value:g}" for value in log_ticks]
below_gate_color = "#8793A1"
last_row = len(groups)

style_figure(fig_dips, "", 1520, 1180)
fig_dips.update_layout(
    margin=dict(l=105, r=38, t=96, b=130),
    legend=dict(
        orientation="h",
        x=0.5, xanchor="center",
        y=-0.085, yanchor="top",
        font=dict(size=14),
    ),
)

for row, g in enumerate(groups, start=1):
    color = profiles[g][1]
    daily = daily_by_mask[g].reindex(plot_dates)
    observed = daily.dropna(subset=["median", "SC"])
    pre = observed.loc[:PRE_EVENT_END]
    qualified = daily["median"].where(daily["SC"].ge(tau))

    # Split at Haiyan so the 4D window stays within each event phase.
    phases = (
        qualified.loc[qualified.index < EVENT_DATE],
        qualified.loc[qualified.index >= EVENT_DATE],
    )
    peak = LOW_NTL
    for phase in phases:
        w4 = phase.rolling(AGGREGATION_DAYS, center=True, min_periods=2).median()
        if w4.notna().any():
            peak = max(peak, w4.max())

        fig_dips.add_trace(go.Scatter(
            x=w4.index, y=w4,
            mode="lines", connectgaps=False, showlegend=False,
            line=dict(color=color, width=1.8, shape="hv"),
            hovertemplate=(
                f"{g} · %{{x|%d %b %Y}}"
                "<br>4D median %{y:.3f}<extra></extra>"
            ),
        ), row=row, col=1)

    for passes in (False, True):
        ink = color if passes else below_gate_color
        symbol = "diamond" if passes else "diamond-open"
        subset = pre.loc[
            pre["SC"].ge(tau).eq(passes) & pre["median"].gt(LOW_NTL)
        ]

        fig_dips.add_trace(go.Scattergl(
            x=subset["SC"], y=subset["median"],
            mode="markers", showlegend=False,
            marker=dict(size=4, opacity=0.38, color=ink),
            text=subset.index.strftime("%d %b %Y"),
            hovertemplate=(
                f"{g} · %{{text}}"
                "<br>SC %{x:.1f}%<br>Daily median %{y:.3f}<extra></extra>"
            ),
        ), row=row, col=2)

        for col, frame in ((1, observed), (2, pre)):
            low = frame.loc[
                frame["SC"].ge(tau).eq(passes) & frame["median"].le(LOW_NTL)
            ]
            y = low["median"] if col == 1 else low["median"].replace(0, zero_pos)

            fig_dips.add_trace(go.Scatter(
                x=low.index if col == 1 else low["SC"],
                y=y, mode="markers", showlegend=False,
                marker=dict(
                    symbol=symbol, size=6.5, color=ink,
                    line=dict(color="#51606D" if passes else ink, width=0.8),
                ),
                text=low.index.strftime("%d %b %Y"),
                customdata=low[["median", "SC", "n"]].to_numpy(),
                hovertemplate=(
                    f"{g} · %{{text}}<br>Daily median %{{customdata[0]:.3f}}"
                    "<br>SC %{customdata[1]:.1f}% · %{customdata[2]:.0f} HQ pixels"
                    "<extra></extra>"
                ),
            ), row=row, col=col)

    add_haiyan_marker(fig_dips, row=row, col=1)
    fig_dips.add_vline(
        x=tau, line=dict(color="#929CA8", width=1, dash="dot"),
        row=row, col=2,
    )

    fig_dips.update_yaxes(
        range=[-0.05 * peak, 1.05 * peak], nticks=4, tickfont=dict(size=12),
        row=row, col=1,
    )
    fig_dips.update_yaxes(
        type="log",
        range=[np.log10(log_bottom) - 0.16, np.log10(log_top) + 0.12],
        tickvals=tickvals, ticktext=ticktext, tickfont=dict(size=12),
        row=row, col=2,
    )
    fig_dips.update_xaxes(
        range=[plot_dates[0], plot_end], dtick="M3", tickformat="%b %Y",
        showticklabels=row == last_row, row=row, col=1,
    )
    fig_dips.update_xaxes(
        range=[-1, 101], tickvals=[0, 25, 50, 75, 100],
        showticklabels=row == last_row, row=row, col=2,
    )

    # Position each GHSL label at the centre of its subplot row.
    row_domain = fig_dips.get_subplot(row, 1).yaxis.domain
    fig_dips.add_annotation(
        x=-0.03,                  # More negative moves the label left
        y=sum(row_domain) / 2,     # Keeps each label centred in its row
        xref="paper", yref="paper",
        text=f"{g}</b>",         # Displays G1, G2, … G8
        showarrow=False,
        xanchor="center", yanchor="middle",
        font=dict(size=14, color=PLOT_TEXT_COLOR),
    )

for col, heading in [
    (1, f"Four-day median NTL ({NTL_UNITS})"),
    (2, f"Pre-Haiyan daily median NTL ({NTL_UNITS})"),
]:
    column_domain = fig_dips.get_subplot(1, col).xaxis.domain
    fig_dips.add_annotation(
        x=sum(column_domain) / 2, y=1.02,
        xref="paper", yref="paper",
        text=f"{heading}</b>", showarrow=False,
        xanchor="center", yanchor="bottom", font=dict(size=16),
    )

fig_dips.add_annotation(
    x=EVENT_DATE, y=1.008, xref="x", yref="paper",
    text="Haiyan", showarrow=False,
    xanchor="left", yanchor="bottom", xshift=5,
    font=dict(color=EVENT_COLOR, size=15),
)

fig_dips.update_xaxes(title_text="Date", row=last_row, col=1)
fig_dips.update_xaxes(title_text="Spatial completeness (%)", row=last_row, col=2)

for name, symbol, ink in [
    (f"Daily median ≤{LOW_NTL:g} · SC ≥{tau:g}%", "diamond", "#43505F"),
    (f"Daily median ≤{LOW_NTL:g} · SC <{tau:g}%", "diamond-open", below_gate_color),
]:
    fig_dips.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers", name=name,
        marker=dict(symbol=symbol, size=10, color=ink),
        hoverinfo="skip",
    ), row=1, col=1)

fig_dips.update_layout(
    margin=dict(l=80, r=22, t=78, b=120),
    legend=dict(
        orientation="v",   # One entry per row
        x=0.1,             # 0 = left, 0.5 = centre, 1 = right
        xanchor="center",
        y=1.05,           # More negative moves it farther down
        yanchor="top",
        font=dict(size=14),
    ),
)

fig_dips.show()

In [12]:
fig_dips.show()
fig_dips.write_image(FIGURE_DIR / ("03_low_radiance_diagnostics" + ".png"), scale=PNG_SCALE)

### Discussion

**The repeated near-zero troughs in G2–G8 are consistent with a lunar-cycle influence on the retrieved radiance, but their cause remains unconfirmed.** The lag analysis shows recurrence near 29–30 days and its multiples, close to the 29.5-day lunar cycle. This differs from the 16-day viewing-geometry pattern seen in the orbital-phase profiles, where typical radiance was higher at more oblique views, including in G1. NASA’s Black Marble algorithm already corrects for reflected moonlight, so the relevant hypothesis is that **residual lunar effects or lunar-dependent retrieval availability** contribute to the troughs. The lag analysis alone cannot identify the lunar phase at which they occur. [ladsweb.modaps.eosdis.nasa.gov](https://ladsweb.modaps.eosdis.nasa.gov/api/v2/content/archives/Document%20Archive/Science%20Data%20Product%20Documentation/VIIRS_Black_Marble_UG_v1.3_Sep_2022.pdf?utm_source=chatgpt.com)

**Settlement composition can explain why the troughs become more frequent as the masks broaden.** G1 is generally brighter and rarely reaches the near-zero threshold. Adding dimmer pixels makes the spatial median more likely to fall near zero, particularly when those pixels make up a larger share of the observations on a given night. The right-hand panels show near-zero medians even at high spatial completeness. This indicates that low coverage alone cannot explain the troughs. Spatial completeness counts how many pixels were retrieved, while the identity and radiance of those pixels can still vary from night to night.
?
A direct test would compare **near-zero frequency across lunar conditions among qualified dates**, using lunar irradiance at overpass as well as viewing zenith angle. Repeating the comparison on a common set of pixels would help distinguish changes in retrieved radiance from changes in which pixels contribute to the median. Until then, the pre-Haiyan troughs establish that **near-zero regional medians can recur under ordinary observation conditions**, which limits their use as stand-alone evidence of a post-Haiyan outage.

In [13]:
# # Run after the viewing-geometry section.
# diagnostic_masks = ("G1", "G3", "G8")
# focus_mask = "G3"
# pre_dates = pd.date_range(CHARACTERIZATION_START, PRE_EVENT_END, name="date")
# tau = SPATIAL_COMPLETENESS_PCT

# # Use the fixed G8 footprint for the regional diagnostic layers.
# diagnostic_pixels = profile_pixels["G8"]
# ay, ax = np.unravel_index(diagnostic_pixels, ghsl.shape)
# sample_y = xr.DataArray(ay, dims="pixel")
# sample_x = xr.DataArray(ax, dims="pixel")

# a1_layers = (
#     "Moon_Illumination_Fraction", "Lunar_Zenith", "Solar_Zenith",
# )
# a2_layers = (
#     "DNB_Lunar_Irradiance", "QF_Cloud_Mask", "Latest_High_Quality_Retrieval",
# )
# layers = pd.DataFrame(np.nan, index=pre_dates, columns=[*a1_layers, *a2_layers])
# layers["VZA"] = regional_vza.reindex(pre_dates)

# # Metadata-free fallbacks assume the local stores already contain physical units.
# # For a verified packed store, set A1 angles/illumination to .01
# # and DNB_Lunar_Irradiance to .1 in this dictionary.
# fallback_scale = {
#     "Moon_Illumination_Fraction": 1.0,
#     "Lunar_Zenith": 1.0,
#     "Solar_Zenith": 1.0,
#     "DNB_Lunar_Irradiance": 1.0,
# }
# upper_limits = {
#     "Moon_Illumination_Fraction": 100,
#     "Lunar_Zenith": 180,
#     "Solar_Zenith": 180,
#     "QF_Cloud_Mask": 65534,
# }

# def read_diagnostic_layers(source, names):
#     available = pd.DatetimeIndex(source.date.values)
#     if "processed" in source:
#         available = available[source["processed"].values == 1]

#     for name in names:
#         if name not in source:
#             continue
#         dates = pre_dates.intersection(available).difference(layers[name].dropna().index)
#         band = source[name]
#         for dim in set(band.dims) - {"date", "y", "x"}:
#             band = band.squeeze(dim)

#         fills = [-32768, -9999, 65535]
#         if name == "Latest_High_Quality_Retrieval":
#             fills.append(255)
#         for key in ("_FillValue", "missing_value"):
#             fills.extend(np.atleast_1d(band.attrs.get(key, [])).tolist())

#         scale = float(band.attrs.get("scale_factor", fallback_scale.get(name, 1.0)))
#         offset = float(band.attrs.get("add_offset", 0.0))
#         limit = upper_limits.get(name, np.inf)

#         for start in range(0, len(dates), 16):
#             batch = dates[start:start + 16]
#             raw = (
#                 band.sel(date=batch).isel(y=sample_y, x=sample_x)
#                 .transpose("date", "pixel").compute().values.astype(float)
#             )
#             raw[~np.isfinite(raw) | np.isin(raw, fills) | (raw < 0)] = np.nan
#             values = raw * scale + offset

#             if name in a1_layers and np.any(values > limit):
#                 raise ValueError(f"{name} exceeds its physical range. Check its stored scale.")
#             values[(values < 0) | (values > limit)] = np.nan

#             if name == "QF_Cloud_Mask":
#                 # Bits 6–7 encode cloud confidence. Values 2 and 3 are cloudy.
#                 flags = np.nan_to_num(values, nan=0).astype(np.uint16)
#                 valid = np.isfinite(values)
#                 cloudy = valid & (((flags >> 6) & 3) >= 2)
#                 counts = valid.sum(axis=1)
#                 summary = 100 * cloudy.sum(axis=1) / np.where(counts > 0, counts, np.nan)
#             else:
#                 summary = median(values, axis=1)

#             layers.loc[batch, name] = summary

# for path in (
#     PROCESSED_DIR / "Haiyan_VNP46A1.zarr",
#     PROCESSED_DIR / "baseline/Haiyan_VNP46A1_baseline.zarr",
# ):
#     if path.exists():
#         with xr.open_zarr(path, chunks={}, consolidated=False, mask_and_scale=False) as ds:
#             for axis in ("x", "y"):
#                 np.testing.assert_allclose(ds[axis], a2[axis], rtol=0, atol=1e-6)
#             read_diagnostic_layers(ds, a1_layers)

# read_diagnostic_layers(a2, a2_layers)
# display(layers.count().rename("Days available").to_frame())

# # 1. Aligned timelines
# illumination = layers["Moon_Illumination_Fraction"]
# if illumination.notna().any():
#     moon_for_plot = illumination
#     moon_label = "A1 Moon illumination"
# else:
#     new_moon = pd.Timestamp("2012-01-23 07:39")
#     lunar_age = ((pre_dates - new_moon) / pd.Timedelta(days=1)) % 29.53059
#     moon_for_plot = pd.Series(50 * (1 - np.cos(2 * np.pi * lunar_age / 29.53059)),
#                               index=pre_dates)
#     moon_label = "Approx. Moon illumination"

# fig_lunar = make_subplots(
#     rows=6, cols=1, shared_xaxes=True, vertical_spacing=.03,
#     specs=[[{"secondary_y": True}], [{}], [{}], [{}], [{}], [{}]],
# )

# def add_daily_line(row, values, name, color, secondary_y=None):
#     if values.notna().any():
#         fig_lunar.add_trace(go.Scatter(
#             x=values.index, y=values, name=name, mode="lines",
#             line=dict(color=color, width=1.3), connectgaps=False,
#             hovertemplate="%{x|%d %b %Y}<br>%{y:.2f}<extra>%{fullData.name}</extra>",
#         ), row=row, col=1, secondary_y=secondary_y)

# add_daily_line(1, moon_for_plot, moon_label, "#7354A3", False)
# add_daily_line(1, layers["DNB_Lunar_Irradiance"], "A2 lunar irradiance", "#C17D15", True)
# add_daily_line(2, layers["VZA"], "Sensor zenith", "#294866")
# add_daily_line(2, layers["Lunar_Zenith"], "Lunar zenith", "#7354A3")
# add_daily_line(2, layers["Solar_Zenith"], "Solar zenith", "#C17D15")
# add_daily_line(3, layers["QF_Cloud_Mask"], "Cloudy pixels", "#778391")
# add_daily_line(4, layers["Latest_High_Quality_Retrieval"], "Latest HQ age", "#52687D")

# for g in diagnostic_masks:
#     daily = daily_by_mask[g].reindex(pre_dates)
#     color = profiles[g][1]
#     add_daily_line(3, daily["SC"], f"{g} SC", color)

#     qualified = daily["median"].where(daily["SC"].ge(tau))
#     add_daily_line(5, qualified.rolling(AGGREGATION_DAYS, center=True, min_periods=2).median(),
#                    f"{g} 4D median", color)

#     # Keep the same denominator throughout the record, including months with no dips.
#     low = qualified.le(LOW_NTL)
#     fig_lunar.add_trace(go.Scatter(
#         x=qualified.index[low], y=[g] * int(low.sum()),
#         mode="markers", name=f"{g} near zero", showlegend=False,
#         marker=dict(symbol="diamond", size=6, color=color),
#         hovertemplate=f"{g} · %{{x|%d %b %Y}}<extra></extra>",
#     ), row=6, col=1)

# style_figure(fig_lunar, "", 1420, 1000)
# fig_lunar.update_layout(
#     margin=dict(l=105, r=90, t=72, b=75),
#     legend=dict(orientation="h", x=0, y=1.02, yanchor="bottom"),
# )
# fig_lunar.update_yaxes(title_text="Moon lit (%)", range=[0, 100], row=1, col=1, secondary_y=False)
# fig_lunar.update_yaxes(title_text="Irradiance (nW cm⁻²)", row=1, col=1, secondary_y=True)
# fig_lunar.update_yaxes(title_text="Zenith (°)", row=2, col=1)
# fig_lunar.update_yaxes(title_text="Coverage (%)", range=[0, 100], row=3, col=1)
# fig_lunar.update_yaxes(title_text="HQ age (days)", row=4, col=1)
# fig_lunar.update_yaxes(title_text=f"4D NTL ({NTL_UNITS})", row=5, col=1)
# fig_lunar.update_yaxes(
#     title_text="Near-zero dates", categoryorder="array",
#     categoryarray=list(reversed(diagnostic_masks)), row=6, col=1,
# )
# fig_lunar.update_xaxes(title_text="Date", row=6, col=1)
# fig_lunar.show()

# # 2. Near-zero frequency by observed lunar illumination and regional VZA
# if illumination.notna().any() and layers["VZA"].notna().any():
#     focus_daily = daily_by_mask[focus_mask].reindex(pre_dates)
#     eligible = focus_daily["SC"].ge(tau) & focus_daily["median"].notna()
#     moon_labels = ["0–<25%", "25–<50%", "50–<75%", "75–100%"]
#     angle_labels = ["<30°", "30–<50°", "≥50°"]
#     moon_bin = pd.cut(illumination, [0, 25, 50, 75, 101], labels=moon_labels, right=False)
#     angle_bin = pd.cut(layers["VZA"], [0, 30, 50, np.inf], labels=angle_labels, right=False)

#     counts = (
#         focus_daily.loc[eligible, "median"].le(LOW_NTL)
#         .groupby([moon_bin.loc[eligible], angle_bin.loc[eligible]], observed=False)
#         .agg(dips="sum", days="size")
#     )
#     dips = counts["dips"].unstack().reindex(index=moon_labels, columns=angle_labels)
#     days = counts["days"].unstack().reindex(index=moon_labels, columns=angle_labels)
#     rate = 100 * dips / days.replace(0, np.nan)

#     # Display counts for every cell. Colour rates with at least five dates.
#     fig_lunar_rate = go.Figure(go.Heatmap(
#         x=angle_labels, y=moon_labels, z=rate.where(days.ge(5)),
#         colorscale="Blues", zmin=0, zmax=100,
#         colorbar=dict(title="Near-zero (%)"),
#     ))
#     for moon in moon_labels:
#         for angle in angle_labels:
#             n = int(days.loc[moon, angle])
#             k = int(dips.loc[moon, angle])
#             fig_lunar_rate.add_annotation(
#                 x=angle, y=moon, text=f"{k}/{n}" if n else "—", showarrow=False,
#                 font=dict(color="white" if n >= 5 and k / n > .55 else "#243B5A"),
#             )

#     style_figure(fig_lunar_rate, "", 780, 410)
#     fig_lunar_rate.update_layout(
#         plot_bgcolor="#E2E6EA", margin=dict(l=105, r=80, t=45, b=70),
#         title=dict(text=f"{focus_mask} · near-zero days / qualified days", x=.5),
#     )
#     fig_lunar_rate.update_xaxes(title_text="G8 median viewing zenith angle")
#     fig_lunar_rate.update_yaxes(title_text="A1 Moon illumination (%)")
#     fig_lunar_rate.show()
# else:
#     print("The joint heatmap requires stored A1 Moon illumination and regional VZA.")

### Recurrence at exact calendar lags

In [14]:
# Pre-Haiyan recurrence of near-zero daily spatial medians.
# Each column is an exact calendar lag. Missing or unqualified dates stay missing.
recurrence_lags = np.arange(1, 91)
pre_dates = pd.date_range(CHARACTERIZATION_START, PRE_EVENT_END)
rows = []

for g in groups:
    daily = daily_by_mask[g].reindex(pre_dates)
    qualified = daily["median"].where(daily.SC.ge(tau))
    low = qualified.le(LOW_NTL).where(qualified.notna()).astype(float)

    for lag in recurrence_lags:
        future = low.shift(-int(lag))
        after_low = future.loc[low.eq(1)].dropna()
        after_other = future.loc[low.eq(0)].dropna()

        rows.append((
            g, lag, len(after_low), len(after_other),
            100 * after_low.mean(), 100 * after_other.mean(),
        ))

recurrence = pd.DataFrame(rows, columns=[
    "mask", "lag", "n_low", "n_other", "after_low_pct", "after_other_pct",
])
recurrence["excess_pp"] = recurrence.after_low_pct - recurrence.after_other_pct

# Display rule only. A single eligible pair would otherwise produce 0% or 100%.
min_pairs = 5
recurrence["shown_pp"] = recurrence.excess_pp.where(
    recurrence.n_low.ge(min_pairs) & recurrence.n_other.ge(min_pairs)
)

def lag_matrix(column):
    return recurrence.pivot(index="mask", columns="lag", values=column).reindex(
        index=groups, columns=recurrence_lags
    )

In [15]:
fig_recurrence = go.Figure(go.Heatmap(
    x=recurrence_lags, y=groups, z=lag_matrix("shown_pp"),
    zmin=-100, zmax=100,
    colorscale=[[0, "#B85D34"], [.5, "#FFFFFF"], [1, "#006AA6"]],
    colorbar=dict(title="Difference<br>(pp)", thickness=13),
    customdata=np.stack([
        lag_matrix("after_low_pct"), lag_matrix("n_low"),
        lag_matrix("after_other_pct"), lag_matrix("n_other"),
    ], axis=-1),
    hoverongaps=False,
    hovertemplate=(
        "%{y} · %{x} calendar days later"
        "<br>After a low day: %{customdata[0]:.1f}% (%{customdata[1]:.0f} pairs)"
        "<br>After another qualified day: %{customdata[2]:.1f}% (%{customdata[3]:.0f} pairs)"
        "<br>Difference: %{z:+.1f} percentage points<extra></extra>"
    ),
))

style_figure(fig_recurrence, "", 1250, 395)
fig_recurrence.update_layout(
    plot_bgcolor="#E2E6EA",
    margin=dict(l=65, r=100, t=15, b=65),
)
fig_recurrence.update_xaxes(title_text="Calendar days between qualified observations",
                 tickvals=[1, 10, 20, 30, 40, 50, 60, 70, 80, 90], showgrid=False)
fig_recurrence.update_yaxes(autorange="reversed", showgrid=False)

### Calendar-lag recurrence of near-zero nighttime lights

The heatmap quantifies the recurrence of near-zero daily nighttime lights (NTL) before Haiyan across the eight cumulative GHSL masks. For each mask and calendar lag, the analysis uses date pairs in which both observations met the 25% spatial completeness threshold. A date is classified as near zero when its daily spatial median is ≤0.05 nW cm⁻² sr⁻¹.

Each cell reports the percentage-point difference in the frequency of near-zero NTL on the later date between pairs starting on a near-zero date and pairs starting above the near-zero threshold. Positive values (blue) indicate higher recurrence after a near-zero observation. Negative values (orange) indicate lower recurrence, and values near zero (white) indicate similar frequencies. Cells are masked in gray when either group contains fewer than five eligible pairs. G1 had only one qualified near-zero date, so recurrence could not be estimated with the required support at any lag.

Positive bands around 29–30, 58–60, and 87–90 days in G2–G8 indicate approximately monthly recurrence among qualified observations. The masks are nested and share pixels, so these patterns provide related evidence across spatial supports. The lag structure identifies the timing of recurrent low observations while leaving the underlying mechanism unresolved.

In [16]:
fig_recurrence.show()
fig_recurrence.write_image(FIGURE_DIR / ("04_pre_haiyan_near_zero_recurrence" + ".png"), scale=PNG_SCALE)

### Discussion

Before Haiyan, near-zero daily medians tended to recur roughly every 29–30 days in G2–G8, even when both dates met the 25% spatial completeness threshold. The clearest evidence is the sequence of blue bands near 29–30, 58–60, and 87–90 days, with lower recurrence between them.

Each row is a settlement mask. To read the plot, take G3 at 30 days. Among pairs of qualified dates exactly 30 calendar days apart, blue means the second date was more likely to have a median ≤0.05 when the first date was also near zero. The color measures that difference in percentage points. Orange means a low starting day was associated with a lower chance of another low at that lag. The gray G1 row means there were too few qualified low-day pairs to assess recurrence.

The important distinction is timing versus spatial composition. Adding darker pixels can make a regional median more likely to cross 0.05. A fixed set of dark pixels, by itself, does not explain why those crossings preferentially recur at roughly monthly intervals. The pattern is already visible in G2, when GHSL class 23 is added to the urban core, and becomes more pronounced in the broader masks. Those masks share pixels, so the repeated bands across rows are related evidence.

### Sensitivity to the spatial completeness threshold

In [17]:
# Pre-Haiyan near-zero recurrence by spatial completeness threshold.
# Daily spatial medians from MQF 0 or 1. No temporal smoothing.

taus = (0, 25, 50)
sensitivity_lags = np.arange(1, 91)
pre_dates = pd.date_range(CHARACTERIZATION_START, PRE_EVENT_END)
min_pairs = 5  # Same display rule as the original recurrence plot.

rows = []
support_rows = []

for g in groups:
    daily = daily_by_mask[g].reindex(pre_dates)
    observed = daily["median"].notna() & daily["n"].gt(0)

    for gate in taus:
        qualified = observed & daily["SC"].ge(gate)
        low = daily["median"].le(LOW_NTL).where(qualified).astype(float)

        n_qualified = int(qualified.sum())
        n_low_days = int(low.eq(1).sum())
        support_rows.append((g, gate, n_qualified, n_low_days))

        for lag in sensitivity_lags:
            # The index contains every calendar day, including missing dates.
            future = low.shift(-int(lag))
            after_low = future.loc[low.eq(1)].dropna()
            after_other = future.loc[low.eq(0)].dropna()

            rows.append((
                g, gate, lag, len(after_low), len(after_other),
                100 * after_low.mean(), 100 * after_other.mean(),
            ))

recurrence_tau = pd.DataFrame(rows, columns=[
    "mask", "tau", "lag", "n_low", "n_other", "after_low_pct", "after_other_pct",
])
recurrence_tau["excess_pp"] = (
    recurrence_tau["after_low_pct"] - recurrence_tau["after_other_pct"]
)
enough_pairs = (
    recurrence_tau["n_low"].ge(min_pairs) & recurrence_tau["n_other"].ge(min_pairs)
)
recurrence_tau["shown_pp"] = recurrence_tau["excess_pp"].where(enough_pairs)

In [18]:
fig_tau = make_subplots(
    rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.035,
    subplot_titles=[f"τ = {gate}%" for gate in taus],
)

for col, gate in enumerate(taus, start=1):
    panel = recurrence_tau.loc[recurrence_tau["tau"].eq(gate)]
    matrices = {
        key: panel.pivot(index="mask", columns="lag", values=key)
                  .reindex(index=groups, columns=sensitivity_lags).to_numpy()
        for key in ["shown_pp", "after_low_pct", "n_low", "after_other_pct", "n_other"]
    }

    fig_tau.add_trace(go.Heatmap(
        x=sensitivity_lags, y=groups, z=matrices["shown_pp"], coloraxis="coloraxis",
        customdata=np.stack([
            matrices["after_low_pct"], matrices["n_low"],
            matrices["after_other_pct"], matrices["n_other"],
        ], axis=-1),
        hoverongaps=False,
        hovertemplate=(
            f"%{{y}} · τ = {gate}% · lag %{{x}} days"
            "<br>Low after low: %{customdata[0]:.1f}% (%{customdata[1]:.0f} pairs)"
            "<br>Low after other: %{customdata[2]:.1f}% (%{customdata[3]:.0f} pairs)"
            "<br>Difference: %{z:+.1f} pp<extra></extra>"
        ),
    ), row=1, col=col)

style_figure(fig_tau, "", 1400, 380)
fig_tau.update_layout(
    plot_bgcolor="#E2E6EA",
    margin=dict(l=55, r=85, t=48, b=60),
    coloraxis=dict(
        cmin=-100, cmax=100,
        colorscale=[[0, "#B85D34"], [.5, "#FFFFFF"], [1, "#006AA6"]],
        colorbar=dict(title="Δ (pp)", thickness=12, len=.85),
    ),
)
fig_tau.update_yaxes(categoryorder="array", categoryarray=groups,
                 autorange="reversed", showgrid=False)
fig_tau.update_xaxes(tickvals=[1, 15, 30, 45, 60, 75, 90], showgrid=False)
fig_tau.update_xaxes(title_text="Calendar lag (days)", row=1, col=2)

In [19]:
fig_tau.show()
fig_tau.write_image(FIGURE_DIR / ("04_pre_haiyan_near_zero_recurrence_by_tau" + ".png"), scale=PNG_SCALE)

In [20]:
# Read alongside the heatmaps, especially where a stricter gate produces gray cells.
recurrence_support = pd.DataFrame(
    support_rows, columns=["mask", "tau", "Qualified days", "Near-zero days"]
)
recurrence_support["Near-zero days (%)"] = (
    100 * recurrence_support["Near-zero days"]
    / recurrence_support["Qualified days"].replace(0, np.nan)
)
display(
    recurrence_support.set_index(["mask", "tau"])
    .unstack("tau").reindex(groups).round(1)
)

Qualified days           Near-zero days         Near-zero days (%)        \
tau              0    25   50             0   25  50                 0     25   
mask                                                                            
G1              399  289  218              6   1   1                1.5   0.3   
G2              482  300  208             41  24  16                8.5   8.0   
G3              501  298  210             66  42  32               13.2  14.1   
G4              510  295  211            106  67  54               20.8  22.7   
G5              521  301  214            117  70  55               22.5  23.3   
G6              527  305  218            135  84  65               25.6  27.5   
G7              530  315  227            150  94  75               28.3  29.8   
G8              531  315  227            151  94  75               28.4  29.8   

            
tau     50  
mask        
G1     0.5  
G2     7.7  
G3    15.2  
G4    25.6  
G5    25.7  
G6    29.8  
G7    33.0  
G8    33.0

### Discussion

At 0% SC, a date still needs one measured pixel. At 25% and 50%, both dates in a lagged pair must meet the gate. Compare a band across thresholds alongside the support table. White indicates a small recurrence difference. Gray indicates insufficient pairs to display the difference.

## 5. Pre-Haiyan lunar illumination, viewing geometry, and nighttime lights

Compare observed VIIRS A1 Moon illumination and viewing zenith angle (VZA) with the four-day nighttime lights (NTL) medians for G1–G3. Solid gray guides mark observed lunar illumination peaks. Diamonds mark individual qualified dates with near-zero NTL, defined as a daily spatial median ≤0.05 nW cm⁻² sr⁻¹. The four-day profiles use at least two qualified daily observations per window.

In [44]:
# Read stored Moon illumination at three G3 pixels nearest the mask centre.
moon_band = "Moon_Illumination_Fraction"
moon = pd.Series(np.nan, index=pre_dates, name=moon_band)
iy, ix = np.unravel_index(profile_pixels["G3"], ghsl.shape)
distance = (iy - np.median(iy)) ** 2 + (ix - np.median(ix)) ** 2
sample = np.argsort(distance)[:3]

for path in (
    PROCESSED_DIR / "baseline/Haiyan_VNP46A1_baseline.zarr",
    PROCESSED_DIR / "Haiyan_VNP46A1.zarr",
):
    if not path.exists():
        continue

    # Decode the stored scale, offset, and fill value.
    with xr.open_zarr(path, chunks=None, consolidated=False, mask_and_scale=True) as ds:
        if moon_band not in ds:
            continue

        for axis in ("x", "y"):
            np.testing.assert_allclose(ds[axis], a2[axis], rtol=0, atol=1e-6)

        available = pd.DatetimeIndex(ds.date.values)
        if "processed" in ds:
            available = available[ds["processed"].values == 1]
        dates = pre_dates.intersection(available).difference(moon.dropna().index)

        band = ds[moon_band]
        extra_dims = [dim for dim in band.dims if dim not in ("date", "y", "x")]
        if extra_dims:
            band = band.squeeze(extra_dims, drop=True)

        for start in tqdm(range(0, len(dates), 16), desc=f"Moon · {path.name}", unit="batch"):
            batch = dates[start:start + 16]
            samples = np.column_stack([
                band.isel(y=int(iy[i]), x=int(ix[i])).sel(date=batch).values
                for i in sample
            ]).astype(float)
            samples[~np.isfinite(samples) | (samples < 0) | (samples == 65535)] = np.nan

            if np.any(samples > 100):
                raise ValueError(
                    f"{path.name}: {moon_band} exceeds 100% after metadata decoding. "
                    "Check the stored scale_factor."
                )

            moon.loc[batch] = median(samples, axis=1)

if moon.dropna().empty:
    raise ValueError("No valid A1 Moon illumination was found at the sampled G3 pixels.")

panels = [("A1 Moon lit (%)", moon, "#7354A3")]

Moon · Haiyan_VNP46A1.zarr: 100%|██████████| 12/12 [00:00<00:00, 43.90batch/s]


In [48]:
from scipy.signal import find_peaks

pre_dates = pd.date_range(CHARACTERIZATION_START, PRE_EVENT_END, name="date")
tau = SPATIAL_COMPLETENESS_PCT
cycle_masks = ("G1", "G2", "G3")
cached_layers = globals().get("layers", pd.DataFrame()).reindex(pre_dates)

# Read the stored Moon band at three G3 pixels near the mask centre.
iy, ix = np.unravel_index(profile_pixels["G3"], ghsl.shape)
distance = (iy - np.median(iy)) ** 2 + (ix - np.median(ix)) ** 2
sample = np.argsort(distance)[:3]

# Reuse regional VZA when available. Otherwise use the same three G3 pixels.
vza = globals().get("regional_vza", pd.Series(dtype=float)).reindex(pre_dates)
use_sampled_vza = vza.dropna().empty
bands = {"Moon_Illumination_Fraction": 100}
if use_sampled_vza:
    bands["Sensor_Zenith"] = 90

a1_daily = pd.DataFrame(np.nan, index=pre_dates, columns=list(bands))
a1_paths = (
    PROCESSED_DIR / "baseline/Haiyan_VNP46A1_baseline.zarr",
    PROCESSED_DIR / "Haiyan_VNP46A1.zarr",
)

for path in a1_paths:
    if not path.exists():
        continue

    with xr.open_zarr(
        path, chunks=None, consolidated=False, mask_and_scale=True,
    ) as ds:
        for axis in ("x", "y"):
            np.testing.assert_allclose(ds[axis], a2[axis], rtol=0, atol=1e-6)

        available = pd.DatetimeIndex(ds.date.values)
        if "processed" in ds:
            available = available[np.asarray(ds["processed"].values) == 1]

        for name, upper_limit in bands.items():
            if name not in ds:
                continue

            dates = pre_dates.intersection(available).difference(
                a1_daily[name].dropna().index
            )
            band = ds[name]
            extra_dims = [dim for dim in band.dims if dim not in ("date", "y", "x")]
            if extra_dims:
                band = band.squeeze(extra_dims, drop=True)

            for start in tqdm(
                range(0, len(dates), 16),
                desc=f"{name} · {path.name}", unit="batch",
            ):
                batch = dates[start:start + 16]
                values = np.column_stack([
                    band.isel(y=int(iy[i]), x=int(ix[i])).sel(date=batch).values
                    for i in sample
                ]).astype(float)
                values[~np.isfinite(values) | (values < 0) | (values == 65535)] = np.nan

                if np.any(values > upper_limit):
                    raise ValueError(
                        f"{path.name}: {name} exceeds {upper_limit} after metadata "
                        "decoding. Check the stored scale_factor."
                    )

                a1_daily.loc[batch, name] = median(values, axis=1)

moon = a1_daily["Moon_Illumination_Fraction"]
if moon.dropna().empty:
    raise ValueError("No valid stored Moon illumination at the sampled G3 pixels.")

if use_sampled_vza:
    vza = a1_daily["Sensor_Zenith"]
    vza_label = "Sampled G3<br>VZA (°)"
else:
    vza_label = "G8 VZA (°)"

panels = [("Moon lit", moon, "#7354A3", None)]
if vza.notna().any():
    panels.append((vza_label, vza, "#294866", None))

# Add daily diagnostics if they are already in memory.
for band, label, color in [
    ("DNB_Lunar_Irradiance", "Lunar irradiance<br>(nW cm⁻²)", "#B87618"),
    ("Lunar_Zenith", "Lunar zenith (°)", "#8769AC"),
    ("Solar_Zenith", "Solar zenith (°)", "#B87618"),
    ("QF_Cloud_Mask", "Cloudy pixels (%)", "#778391"),
]:
    if band in cached_layers and cached_layers[band].notna().any():
        panels.append((label, cached_layers[band], color, None))

for g in cycle_masks:
    daily = daily_by_mask[g].reindex(pre_dates)
    qualified = daily["median"].where(daily["SC"].ge(tau))
    ntl_4d = qualified.rolling(
        AGGREGATION_DAYS, center=True, min_periods=2,
    ).median()
    dips = qualified.loc[qualified.le(LOW_NTL)]
    panels.append((f"{g} · 4D NTL<br>({NTL_UNITS})", ntl_4d, profiles[g][1], dips))

fig_cycles = make_subplots(
    rows=len(panels), cols=1, shared_xaxes=True, vertical_spacing=.025,
)

style_figure(fig_cycles, "", 1400, 135 * len(panels) + 75)
fig_cycles.update_layout(
    margin=dict(l=175, r=25, t=25, b=55),
    hovermode="x",
    showlegend=False,
)
fig_cycles.update_xaxes(showgrid=False, zeroline=False)
fig_cycles.update_yaxes(
    showgrid=False, zeroline=False, nticks=3, tickformat=".2~f",
)

for row, (label, series, color, dips) in enumerate(panels, start=1):
    fig_cycles.add_trace(go.Scatter(
        x=series.index, y=series, name=label, mode="lines",
        line=dict(color=color, width=1.5, shape="hv" if dips is not None else "linear"),
        connectgaps=False,
        hovertemplate="%{x|%d %b %Y}<br>%{y:.2f}<extra>%{fullData.name}</extra>",
    ), row=row, col=1)

    if dips is not None:
        fig_cycles.add_trace(go.Scatter(
            x=dips.index, y=dips, mode="markers",
            name="Qualified near-zero day",
            marker=dict(symbol="diamond", size=6, color=color),
            hovertemplate=(
                "%{x|%d %b %Y}<br>Daily median %{y:.3f}"
                "<extra>Qualified near-zero day</extra>"
            ),
        ), row=row, col=1)

    domain = fig_cycles.get_subplot(row, 1).yaxis.domain
    fig_cycles.add_annotation(
        x=-0.055, y=sum(domain) / 2,
        xref="paper", yref="paper",
        xanchor="right", yanchor="middle",
        text=label, align="right", showarrow=False, font=dict(size=13),
    )

# Mark well-separated peaks directly from the stored lunar curve.
# Observation gaps are left unfilled.
for _, segment in moon.dropna().groupby(moon.isna().cumsum()):
    peaks, _ = find_peaks(
        segment.to_numpy(), height=90, distance=20, prominence=20,
    )
    for date in segment.index[peaks]:
        fig_cycles.add_shape(
            type="line",
            x0=date, x1=date, xref="x",
            y0=0, y1=1, yref="paper",
            line=dict(color="#B8BDC4", width=1, dash="solid"),
            layer="below",
        )

fig_cycles.update_yaxes(
    range=[0, 105], tickvals=[0, 50, 100], ticksuffix="%", row=1, col=1,
)
fig_cycles.update_xaxes(title_text="Date", row=len(panels), col=1)
fig_cycles.show()

Moon_Illumination_Fraction · Haiyan_VNP46A1_baseline.zarr: 100%|██████████| 30/30 [00:00<00:00, 30.13batch/s]
Moon_Illumination_Fraction · Haiyan_VNP46A1.zarr: 100%|██████████| 12/12 [00:00<00:00, 44.38batch/s]


### 5.1 Discussion

Several qualified near-zero observations in G2 and G3 occur close to peaks in measured lunar illumination. G1 seldom reaches the same radiance threshold. The timing is consistent with the approximately monthly recurrence found in the lag analysis. The greater frequency in G2 and G3 suggests that the response of the spatial median depends on settlement composition. Adding dimmer pixels makes the median more sensitive to changes in their retrieved radiance or in which pixels are observed.

Viewing zenith angle varies on a shorter timescale. Earlier orbital-phase profiles showed that typical NTL was higher at more oblique views, including in G1. This supports an association between viewing geometry and the usual radiance level. The clustering of some G2–G3 near-zero observations around lunar illumination peaks suggests an additional source of variation. The figure shows temporal alignment, while the relative contributions of lunar conditions, viewing geometry, and retrieval availability remain unresolved.

The gray guides are visual references derived from the measured A1 curve. A gap near an illumination maximum can leave a lunation without a guide. To quantify the lunar association, compare the fraction of qualified dates with near-zero NTL across illumination or irradiance ranges within comparable VZA ranges. Use the daily NTL medians for that comparison because the overlapping four-day windows can extend the visual duration of a dip.

## 6. Pre-Haiyan orbital-phase structure in nighttime lights

Compare daily viewing zenith angle (VZA) and four-day nighttime lights (NTL) profiles across the eight cumulative GHSL masks. The left column shows the calendar record through January 2014. Diamonds mark individual near-zero daily medians, with fill indicating whether the date met the spatial completeness threshold. The VZA panel marks the qualified G3 near-zero dates for visual comparison.

The right column groups pre-Haiyan daily observations by position in the 16-day orbital cycle. Points show the median and whiskers span the middle 50% of observations. NTL summaries use qualified daily spatial medians. The VZA summary uses all available G8 viewing angles, regardless of NTL qualification. Each point requires at least three dates.

In [58]:
tau = SPATIAL_COMPLETENESS_PCT
focus = "G3"
cycle_days = 16

pre_dates = pd.date_range(CHARACTERIZATION_START, PRE_EVENT_END, name="date")
phase_day = pd.Series(
    (pre_dates - pre_dates[0]).days % cycle_days + 1,
    index=pre_dates,
)
phase_index = pd.RangeIndex(1, cycle_days + 1)

def orbital_phase_summary(series):
    by_day = series.reindex(pre_dates).groupby(phase_day)
    return pd.DataFrame({
        "median": by_day.median(),
        "q25": by_day.quantile(.25),
        "q75": by_day.quantile(.75),
        "n": by_day.count(),
    }).reindex(phase_index)

vza = regional_vza.reindex(plot_dates)
focus_daily = daily_by_mask[focus].reindex(plot_dates)
focus_dip = focus_daily["median"].le(LOW_NTL) & focus_daily["SC"].ge(tau)
vza_on_dips = vza.where(focus_dip).dropna()

n_rows = len(groups) + 1
fig_cycle = make_subplots(
    rows=n_rows, cols=2, shared_xaxes="columns",
    column_widths=[.72, .28],
    horizontal_spacing=.075, vertical_spacing=.022,
    row_heights=[1.15] + [1] * len(groups),
)

# Left column · daily VZA and qualified G3 dip dates.
fig_cycle.add_trace(go.Scatter(
    x=vza.index, y=vza, mode="lines", connectgaps=False, showlegend=False,
    line=dict(color="#52687D", width=1.25),
    hovertemplate="%{x|%d %b %Y}<br>G8 VZA %{y:.1f}°<extra></extra>",
), row=1, col=1)

fig_cycle.add_trace(go.Scatter(
    x=vza_on_dips.index, y=vza_on_dips, mode="markers", showlegend=False,
    marker=dict(symbol="diamond", size=6, color=profiles[focus][1]),
    hovertemplate=(
        "%{x|%d %b %Y}<br>G8 VZA %{y:.1f}°"
        f"<extra>Qualified {focus} dip</extra>"
    ),
), row=1, col=1)

# Left NTL timelines and right orbital-phase summaries.
for row, label in enumerate(["VZA"] + list(groups), start=1):
    if row == 1:
        color = "#52687D"
        values = vza
        row_label = f"G8 VZA (°)<br>◆ {focus} dip"
        timeline_max = vza.max()
    else:
        frame = daily_by_mask[label].reindex(plot_dates)
        color = profiles[label][1]
        values = frame["median"].where(frame["SC"].ge(tau))
        row_label = f"{label}</b>"
        timeline_max = LOW_NTL

        # Keep centred windows on their own side of Haiyan.
        for segment in (values.loc[:PRE_EVENT_END], values.loc[EVENT_DATE:]):
            smooth = segment.rolling(
                AGGREGATION_DAYS, center=True, min_periods=2,
            ).median()
            if smooth.notna().any():
                timeline_max = max(timeline_max, smooth.max())

            fig_cycle.add_trace(go.Scatter(
                x=smooth.index, y=smooth, mode="lines",
                connectgaps=False, showlegend=False,
                line=dict(color=color, width=1.6, shape="hv"),
                hovertemplate=(
                    f"{label} · %{{x|%d %b %Y}}"
                    "<br>4D median %{y:.3f}<extra></extra>"
                ),
            ), row=row, col=1)

        for passed in (False, True):
            low = frame.loc[
                frame["SC"].notna()
                & frame["SC"].ge(tau).eq(passed)
                & frame["median"].le(LOW_NTL)
            ]
            fig_cycle.add_trace(go.Scatter(
                x=low.index, y=low["median"], mode="markers", showlegend=False,
                marker=dict(
                    symbol="diamond" if passed else "diamond-open",
                    color=color if passed else "#8793A1", size=5.5,
                ),
                customdata=low["SC"],
                hovertemplate=(
                    f"{label} · %{{x|%d %b %Y}}"
                    "<br>Daily median %{y:.3f}<br>SC %{customdata:.1f}%"
                    "<extra></extra>"
                ),
            ), row=row, col=1)

    part = orbital_phase_summary(values)
    mid = part["median"].where(part["n"].ge(3))

    fig_cycle.add_trace(go.Scatter(
        x=part.index, y=mid, mode="markers", showlegend=False,
        marker=dict(size=6, color=color),
        error_y=dict(
            type="data", symmetric=False,
            array=part["q75"] - mid,
            arrayminus=mid - part["q25"],
            color=color, thickness=.9, width=2,
        ),
        customdata=part[["q25", "q75", "n"]],
        hovertemplate=(
            f"{label} · cycle day %{{x}}<br>Median %{{y:.3f}}"
            "<br>Middle 50% %{customdata[0]:.3f}–%{customdata[1]:.3f}"
            "<br>%{customdata[2]:.0f} dates<extra></extra>"
        ),
    ), row=row, col=2)

    # Use one scale per row, including the displayed upper quartile.
    upper = max(
        LOW_NTL,
        timeline_max if pd.notna(timeline_max) else LOW_NTL,
        part.loc[mid.notna(), "q75"].max() if mid.notna().any() else LOW_NTL,
    )
    for col in (1, 2):
        fig_cycle.update_yaxes(
            range=[-.04 * upper, 1.07 * upper],
            nticks=3, tickformat=".2~f", tickfont=dict(size=12),
            row=row, col=col,
        )

    domain = fig_cycle.get_subplot(row, 1).yaxis.domain
    fig_cycle.add_annotation(
        x=-.03, y=sum(domain) / 2, xref="paper", yref="paper",
        text=row_label, showarrow=False,
        xanchor="right", yanchor="middle", align="right",
        font=dict(size=16, color=PLOT_TEXT_COLOR),
    )
    add_haiyan_marker(fig_cycle, row=row, col=1)

style_figure(fig_cycle, "", 1520, 1250)

fig_cycle.update_xaxes(
    range=[plot_dates[0], plot_dates[-1]],
    tickformat="%b %Y", dtick="M3", col=1,
)
fig_cycle.update_xaxes(
    range=[.5, cycle_days + .5],
    tickvals=[1, 4, 8, 12, 16], col=2,
)
fig_cycle.update_xaxes(title_text="Date", row=n_rows, col=1)
fig_cycle.update_xaxes(title_text="Day in the 16-day cycle", row=n_rows, col=2)

for col, heading in [
    (1, f"Calendar record · 4D NTL ({NTL_UNITS})"),
    (2, f"Pre-Haiyan orbital profile · daily NTL ({NTL_UNITS})"),
]:
    domain = fig_cycle.get_subplot(1, col).xaxis.domain
    fig_cycle.add_annotation(
        x=sum(domain) / 2, y=1.055, xref="paper", yref="paper",
        text=f"{heading}</b>", showarrow=False,
        xanchor="center", yanchor="bottom", font=dict(size=16),
    )

fig_cycle.add_annotation(
    x=EVENT_DATE, xref="x", y=1.008, yref="paper",
    text="Haiyan", showarrow=False,
    xanchor="left", yanchor="bottom", xshift=5,
    font=dict(size=14, color=EVENT_COLOR),
)

for label, symbol, color in [
    (f"Daily median ≤{LOW_NTL:g} · SC ≥{tau:g}%", "diamond", "#43505F"),
    (f"Daily median ≤{LOW_NTL:g} · SC <{tau:g}%", "diamond-open", "#8793A1"),
]:
    fig_cycle.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers", name=label,
        marker=dict(symbol=symbol, size=7, color=color),
        hoverinfo="skip",
    ), row=1, col=1)

fig_cycle.update_layout(
    margin=dict(l=150, r=22, t=78, b=120),
    legend=dict(
        orientation="v",   # One entry per row
        x=0.1,             # 0 = left, 0.5 = centre, 1 = right
        xanchor="center",
        y=1.05,           # More negative moves it farther down
        yanchor="top",
        font=dict(size=16),
    ),
)

fig_cycle.show()

In [60]:
pre_dates = pd.date_range(CHARACTERIZATION_START, PRE_EVENT_END, name="date")
cycle_masks = tuple(groups)  # Use ("G1", "G2", "G3") for a shorter figure.
tau = SPATIAL_COMPLETENESS_PCT
n_bins = 16
bin_index = pd.RangeIndex(1, n_bins + 1)

# Orbital day 1 is the first date of the characterisation period.
orbital_day = pd.Series(
    (pre_dates - pre_dates[0]).days % n_bins + 1, index=pre_dates,
)
vza_profile = regional_vza.reindex(pre_dates)

# Infer waxing and waning from nearby A1 observations.
# Keep only dates with observations on both sides within three calendar days.
moon_profile = moon.reindex(pre_dates)
observed_moon = moon_profile.dropna()
observed_dates = observed_moon.index.to_series()
nearby = (
    observed_dates.diff().le(pd.Timedelta(days=3))
    & (observed_dates.shift(-1) - observed_dates).le(pd.Timedelta(days=3))
)
illumination_change = observed_moon.shift(-1) - observed_moon.shift(1)

# Map measured illumination to fractional phase, then distinguish the two halves.
# 0 = new Moon, 0.5 = full Moon, 1 = next new Moon.
half_phase = np.arccos(1 - 2 * observed_moon.clip(0, 100) / 100) / (2 * np.pi)
lunar_phase = half_phase.where(illumination_change.ge(0), 1 - half_phase)
lunar_phase = lunar_phase.where(nearby & illumination_change.ne(0)).reindex(pre_dates)

lunar_bin = pd.cut(
    lunar_phase, bins=np.linspace(0, 1, n_bins + 1),
    labels=bin_index, include_lowest=True,
)
lunar_centres = (np.arange(n_bins) + .5) / n_bins

# Use common dates in the two columns so missing geometry does not
# change which NTL observations contribute to each comparison.
common_support = vza_profile.notna() & lunar_phase.notna()

def summarise_profile(values, bins):
    grouped = values.groupby(bins, observed=True)
    summary = grouped.agg(median="median", n="count")
    summary["q25"] = grouped.quantile(.25)
    summary["q75"] = grouped.quantile(.75)
    return summary.reindex(bin_index)

fig_cycle = make_subplots(
    rows=len(cycle_masks) + 1, cols=2, shared_xaxes="columns",
    horizontal_spacing=.10, vertical_spacing=.025,
    column_titles=["16-day orbital profile", "Lunar-cycle profile"],
)

row_series = [
    ("", vza_profile, moon_profile, "#52687D", "#7354A3"),
]
for g in cycle_masks:
    daily = daily_by_mask[g].reindex(pre_dates)
    ntl = daily["median"].where(daily["SC"].ge(tau))
    color = profiles[g][1]
    row_series.append((g, ntl, ntl, color, color))

for row, (label, left, right, left_color, right_color) in enumerate(row_series, start=1):
    upper_quartiles = []

    for col, values, bins, x, color in [
        (1, left, orbital_day, bin_index, left_color),
        (2, right, lunar_bin, lunar_centres, right_color),
    ]:
        values = values.where(common_support)
        part = summarise_profile(values, bins)
        mid = part["median"].where(part["n"].ge(3))
        upper_quartiles.append(part.loc[mid.notna(), "q75"].max())

        if row == 1:
            hover = (
                "Median %{y:.1f}<br>IQR %{customdata[0]:.1f}–%{customdata[1]:.1f}"
                "<br>%{customdata[2]:.0f} dates<extra></extra>"
            )
        else:
            part["dips"] = (
                values.le(LOW_NTL).where(values.notna())
                .groupby(bins, observed=True).sum().reindex(bin_index)
            )
            hover = (
                f"{label}<br>Median %{{y:.3f}}"
                "<br>IQR %{customdata[0]:.3f}–%{customdata[1]:.3f}"
                "<br>Near zero %{customdata[3]:.0f} / %{customdata[2]:.0f} dates"
                "<extra></extra>"
            )

        fig_cycle.add_trace(go.Scatter(
            x=x, y=mid, mode="lines+markers", connectgaps=False,
            line=dict(color=color, width=1.2),
            marker=dict(color=color, size=5),
            error_y=dict(
                type="data", symmetric=False,
                array=part["q75"] - mid, arrayminus=mid - part["q25"],
                color=color, thickness=.8, width=2,
            ),
            customdata=part[["q25", "q75", "n"] + (["dips"] if row > 1 else [])],
            hovertemplate=hover,
            showlegend=False,
        ), row=row, col=col)

    if row == 1:
        fig_cycle.update_yaxes(title_text="G8 VZA (°)", row=row, col=1)
        fig_cycle.update_yaxes(
            title_text="Moon lit (%)", range=[0, 105], tickvals=[0, 50, 100],
            row=row, col=2,
        )
    else:
        peak = max([LOW_NTL] + [value for value in upper_quartiles if pd.notna(value)])
        for col in (1, 2):
            fig_cycle.update_yaxes(range=[-.04 * peak, 1.08 * peak], row=row, col=col)

        domain = fig_cycle.get_subplot(row, 1).yaxis.domain
        fig_cycle.add_annotation(
            x=-.065, y=sum(domain) / 2, xref="paper", yref="paper",
            text=f"<b>{label}</b>", xanchor="right", yanchor="middle",
            showarrow=False, font=dict(size=13, color=PLOT_TEXT_COLOR),
        )

style_figure(fig_cycle, "", 1120, 105 * len(row_series) + 130)
fig_cycle.update_layout(margin=dict(l=120, r=30, t=65, b=85), hovermode="closest")
fig_cycle.update_yaxes(nticks=3, tickformat=".2~f", showgrid=False, zeroline=False)
fig_cycle.update_xaxes(
    range=[.5, 16.5], tickvals=[1, 4, 8, 12, 16], showgrid=False, col=1,
)
fig_cycle.update_xaxes(
    range=[-.025, 1.025], tickvals=[0, .25, .5, .75, 1],
    ticktext=["New", "First quarter", "Full", "Last quarter", "New"],
    showgrid=False, col=2,
)
fig_cycle.update_xaxes(title_text="Orbital day", row=len(row_series), col=1)
fig_cycle.update_xaxes(title_text="Lunar phase", row=len(row_series), col=2)

fig_cycle.add_annotation(
    x=-.10, y=.43, xref="paper", yref="paper",
    text=f"Daily NTL ({NTL_UNITS})", textangle=-90,
    showarrow=False, font=dict(size=13, color=PLOT_TEXT_COLOR),
)

fig_cycle.update_layout(
    width=1520,
    margin=dict(l=155, r=35, t=65, b=85),
)

# Dark median profiles. Each trace keeps its existing error-bar color.
fig_cycle.update_traces(
    line=dict(color="#253344", width=1.5),
    marker=dict(color="#253344", size=4),
    selector=dict(type="scatter"),
)

# Light guides at the labelled orbital days and lunar phases.
fig_cycle.update_xaxes(showgrid=True, gridcolor="#E2E7ED", gridwidth=1)
fig_cycle.update_yaxes(showgrid=True, gridcolor="#EDF0F3", gridwidth=1)

# Use separate positions for the GHSL labels and the shared radiance label.
ntl_top = fig_cycle.get_subplot(2, 1).yaxis.domain[1]
ntl_bottom = fig_cycle.get_subplot(len(row_series), 1).yaxis.domain[0]

for annotation in fig_cycle.layout.annotations:
    label = annotation.text.replace("<b>", "").replace("</b>", "")

    if label in cycle_masks:
        annotation.update(
            text=label, x=-.042, xanchor="right",
            font=dict(size=13, color=PLOT_TEXT_COLOR),
        )
    elif label.startswith("Daily NTL ("):
        annotation.update(
            x=-.090, y=(ntl_top + ntl_bottom) / 2,
            xanchor="center", yanchor="middle",
        )

fig_cycle.show()

### 6.1 Discussion

The orbital and lunar profiles show different patterns across the GHSL masks. In G1, median radiance varies more across orbital day than across lunar phase. Several G1 radiance peaks coincide with the more oblique G8 viewing angles, consistent with an association between viewing geometry and typical urban-core brightness.

The lunar pattern becomes more pronounced as the masks broaden. G2 shows a clear trough near full Moon, while G3–G8 reach near-zero phase medians and brighten again toward new Moon. Because these profiles summarise qualified daily observations, the full-Moon trough represents a shift in the median of the contributing dates. It therefore extends the earlier result that individual near-zero observations recur at roughly monthly intervals.

The two columns use the same qualified dates within each mask. The set of observed pixels can still change from date to date, and the nested masks share pixels. The figure supports a lunar-phase association in the broader masks, with residual lunar effects and changes in which pixels are retrieved both plausible explanations. The top-right illumination curve serves as a phase reference because the phase coordinate is itself inferred from measured illumination.

For the pre-Haiyan baseline, viewing geometry is particularly relevant when comparing G1 radiance across dates. In the broader masks, a near-zero median around full Moon also falls within the observed pre-event pattern, which limits its value as stand-alone evidence of a post-Haiyan outage.

In [24]:
g1_pixels = land_pixels
regional_vza = pd.Series(np.nan, index=plot_dates, name="G8 VZA")
zenith_values = np.full(pixel_values.shape, np.nan, dtype="float32")
a1_seen = pd.Series(False, index=daily_calendar)
# The baseline store was displayed in degrees. Set the fallback to .01 for
# a metadata-free store known to contain packed centidegrees.
a1_stores = [
    (PROCESSED_DIR / "Haiyan_VNP46A1.zarr", 1.0),
    (PROCESSED_DIR / "baseline/Haiyan_VNP46A1_baseline.zarr", 1.0),
]
for path, fallback_scale in a1_stores:
    if not path.exists():
        print(f"Missing A1 store: {path.name}")
        continue
    with xr.open_zarr(path, chunks={}, consolidated=False, mask_and_scale=False) as ds:
        for axis in ("x", "y"):
            np.testing.assert_allclose(ds[axis], a2[axis], rtol=0, atol=1e-6)
        band = ds["Sensor_Zenith"]
        for dim in set(band.dims) - {"date", "x", "y"}:
            band = band.squeeze(dim)
        scale = float(band.attrs.get("scale_factor", fallback_scale))
        fills = [-32768, -9999, 65535]
        for key in ("_FillValue", "missing_value"):
            fills.extend(np.atleast_1d(band.attrs.get(key, [])).tolist())
        available = pd.DatetimeIndex(ds.date.values)
        if "processed" in ds:
            available = available[ds.processed.values == 1]
        selected = plot_dates.intersection(available).difference(regional_vza.dropna().index)
        for start in tqdm(range(0, len(selected), 8), desc=path.name):
            dates_batch = selected[start:start + 8]
            raw = band.sel(date=dates_batch).transpose("date", "y", "x").compute().values
            angles = raw.reshape(len(dates_batch), -1)[:, plot_pixels].astype("float32")
            angles[np.isin(angles, fills) | (angles < 0)] = np.nan
            if "scale_factor" not in band.attrs and scale == 1 and np.any((angles > 90) & (angles <= 9000)):
                raise ValueError(f"Check the angle encoding for {path.name} and set its fallback scale above.")
            angles = angles * scale + float(band.attrs.get("add_offset", 0))
            angles[(angles < 0) | (angles > 90)] = np.nan
            regional_vza.loc[dates_batch] = median(angles, axis=1)
            pre = dates_batch <= PRE_EVENT_END
            zenith_values[daily_calendar.get_indexer(dates_batch[pre])] = angles[pre][:, mask_cols["G1"]]
            a1_seen.loc[dates_batch[pre]] = True
valid_vza = np.isfinite(zenith_values)
pair = valid_vza & np.isfinite(pixel_values)
paired_ntl = np.where(pair, pixel_values, np.nan)
paired_day = pd.DataFrame({
    "n_vza": valid_vza.sum(axis=1), "n_paired": pair.sum(axis=1),
    "all_vza_median": median(zenith_values, axis=1),
    "median_vza": median(np.where(pair, zenith_values, np.nan), axis=1),
    "median_ntl": median(paired_ntl, axis=1),
    "SC": 100 * pair.mean(axis=1), "A2_SC": primary.SC,
}, index=daily_calendar)
paired_day["A1_SC"] = (100 * valid_vza.mean(axis=1))
paired_day.loc[~a1_seen, "A1_SC"] = np.nan
paired_day.loc[~a1_seen | primary.SC.isna(), "SC"] = np.nan
paired_day.to_csv(TABLE_DIR / "pre_haiyan_matched_vza_daily.csv")

Haiyan_VNP46A1_baseline.zarr: 100%|██████████| 60/60 [00:01<00:00, 57.27it/s]


### G1 overpass angles and paired radiance

In [25]:
fig_geometry = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.56, 0.44], vertical_spacing=0.07,
)

# All valid A1 pixel angles, regardless of A2 availability or qualification.
day_idx, pixel_idx = np.nonzero(valid_vza)
fig_geometry.add_trace(go.Scattergl(
    x=daily_calendar.take(day_idx),
    y=zenith_values[day_idx, pixel_idx],
    mode="markers", name="G1 pixels",
    marker=dict(size=3, color="#8793A1", opacity=0.16),
    hoverinfo="skip",
), row=1, col=1)

# Keep the complete calendar so missing A1 days break the line.
vza_daily = paired_day["all_vza_median"].reindex(daily_calendar)
fig_geometry.add_trace(go.Scatter(
    x=vza_daily.index, y=vza_daily,
    mode="lines+markers", name="Median VZA", connectgaps=False,
    line=dict(color="#294866", width=1.2),
    marker=dict(size=3, color="#294866"),
    customdata=paired_day["n_vza"].reindex(daily_calendar),
    hovertemplate=(
        "%{x|%d %b %Y}<br>Median VZA %{y:.2f}°"
        "<br>%{customdata:.0f} G1 pixels<extra></extra>"
    ),
), row=1, col=1)

for passes, symbol, color in [
    (False, "x", "#AEB7C1"),
    (True, "circle", PRIMARY_COLOR),
]:
    part = paired_day.loc[
        paired_day["median_ntl"].notna()
        & paired_day["SC"].ge(tau).eq(passes)
    ]
    label = f"Paired SC ≥{tau:g}%" if passes else f"Paired SC <{tau:g}%"
    fig_geometry.add_trace(go.Scatter(
        x=part.index, y=part["median_ntl"],
        mode="markers", name=label,
        marker=dict(symbol=symbol, color=color, size=5, opacity=0.7),
        customdata=part[["median_vza", "SC", "n_paired"]],
        hovertemplate=(
            "%{x|%d %b %Y}<br>Median NTL %{y:.3f}"
            "<br>Matched VZA %{customdata[0]:.2f}°"
            "<br>Paired SC %{customdata[1]:.1f}% · %{customdata[2]:.0f} pixels"
            "<extra></extra>"
        ),
    ), row=2, col=1)

style_figure(fig_geometry, "", 1420, 630)
fig_geometry.update_layout(margin=dict(l=90, r=35, t=55, b=95))
fig_geometry.update_yaxes(title_text="G1 VZA (°)", rangemode="tozero", row=1, col=1)
fig_geometry.update_yaxes(title_text=f"Median NTL ({NTL_UNITS})", rangemode="tozero", row=2, col=1)
fig_geometry.update_xaxes(
    range=[daily_calendar.min(), daily_calendar.max()],
    rangeselector=dict(
        x=0, y=1.02,
        buttons=[
            dict(count=32, label="32 days", step="day", stepmode="backward"),
            dict(step="all", label="Full period"),
        ],
    ),
    row=1, col=1,
)

In [26]:
fig_geometry.show()
fig_geometry.write_image(FIGURE_DIR / ("05_paired_viewing_geometry" + ".png"), scale=PNG_SCALE)

In [27]:
# Compare the nominal repeat against adjacent lags and the NTL recurrence lags.
repeat_rows = []
for lag in [1, 15, 16, 17, 29, 30, 32]:
    difference = (vza_daily - vza_daily.shift(lag)).abs().dropna()
    repeat_rows.append((lag, len(difference), difference.median()))

vza_repeat = pd.DataFrame(repeat_rows, columns=[
    "Lag (days)", "Paired dates", "Median absolute VZA difference (°)",
]).set_index("Lag (days)")
display(vza_repeat.round(2))

,Paired dates,Median absolute VZA difference (°)
Lag (days),,
1,647,20.250000
15,631,20.200001
16,630,0.140000
17,629,20.219999
29,617,31.969999
30,616,31.510000
32,616,0.240000


### Discussion

The geometry panel shows repeated changes from near-nadir to oblique views even during gaps in qualified radiance. Read the repeat table in degrees. Small differences at 16 and 32 days relative to adjacent lags support a repeatable viewing sequence. The lower panel uses the intersection of valid A1 angles and HQ A2 radiance.

### G1–G8 profiles by cycle day

The left column keeps chronological four-day profiles. The right column groups unsmoothed pre-event observations by cycle day. Whiskers span the middle 50% of contributing dates. The 29.5-day option uses elapsed time from the first date and has no astronomical phase calibration.

In [28]:
cycle_view = "16-day"  # Also "28-day", "29.5-day", or "calendar month".
focus = "G3"
pre_dates = pd.date_range(CHARACTERIZATION_START, PRE_EVENT_END, name="date")
elapsed = (pre_dates - pre_dates[0]).days.to_numpy()

if cycle_view == "calendar month":
    phase = pd.Series(pre_dates.day, index=pre_dates)
    phase_days = np.arange(1, 32)
    phase_label = "Day of month"
else:
    period = {"16-day": 16, "28-day": 28, "29.5-day": 29.5}[cycle_view]
    phase = pd.Series(np.floor(elapsed % period).astype(int) + 1, index=pre_dates)
    phase_days = np.arange(1, int(np.ceil(period)) + 1)
    phase_label = f"Day in {period:g}-day cycle"

# For the fractional period, the last bin spans half a day. Hover reports support.
# Phase day 1 is the first pre-event calendar date, not a named orbital or lunar phase.
phase_ticks = [d for d in (1, 5, 10, 15, 20, 25, 30) if d <= phase_days[-1]]
if phase_days[-1] - phase_ticks[-1] >= 3:
    phase_ticks.append(int(phase_days[-1]))

def phase_summary(values):
    grouped = values.reindex(pre_dates).groupby(phase)
    return pd.DataFrame({
        "q25": grouped.quantile(.25),
        "median": grouped.median(),
        "q75": grouped.quantile(.75),
        "n": grouped.count(),
    }).reindex(phase_days)

In [29]:
fig_cycle = make_subplots(rows=9, cols=2, shared_xaxes="columns",
    column_widths=[.76, .24], horizontal_spacing=.075, vertical_spacing=.017,
    row_heights=[.16] + [.105] * 8)

# Full timeline on the left. Direct row/column addressing on the right.
fig_cycle.add_trace(go.Scatter(x=plot_dates, y=regional_vza, mode="lines",
    line=dict(color="#73869A", width=1), name="G8 VZA"), row=1, col=1)
focus_data = daily_by_mask[focus]
dip = focus_data["median"].le(LOW_NTL) & focus_data.SC.ge(tau)
fig_cycle.add_trace(go.Scatter(x=plot_dates[dip], y=regional_vza[dip], mode="markers",
    marker=dict(symbol="diamond", size=5, color="#283747"), name=f"{focus} dip"), row=1, col=1)
for row, label in enumerate(["VZA"] + groups, start=1):
    if row == 1:
        color = "#52687D"
        values = regional_vza
    else:
        frame = daily_by_mask[label]
        color = profiles[label][1]
        values = frame["median"].where(frame.SC.ge(tau))
        peak = LOW_NTL
        for segment in (values.loc[:PRE_EVENT_END], values.loc[EVENT_DATE:]):
            smooth = segment.rolling(AGGREGATION_DAYS, center=True, min_periods=2).median()
            peak = max(peak, smooth.max())
            fig_cycle.add_trace(go.Scatter(x=smooth.index, y=smooth, mode="lines",
                line=dict(color=color, width=1.6, shape="hv"), connectgaps=False), row=row, col=1)
        for passed in (False, True):
            low = frame.loc[frame["median"].le(LOW_NTL) & frame.SC.ge(tau).eq(passed)]
            fig_cycle.add_trace(go.Scatter(x=low.index, y=low["median"], mode="markers",
                marker=dict(symbol="diamond" if passed else "diamond-open",
                            color=color if passed else "#8793A1", size=5)), row=row, col=1)
        fig_cycle.update_yaxes(range=[0, peak], row=row, col=1)
    part = phase_summary(values)
    mid = part["median"].where(part.n.ge(3))
    fig_cycle.add_trace(go.Scatter(x=part.index, y=mid, mode="lines+markers",
        line=dict(color=color, width=1.6), marker=dict(size=4),
        error_y=dict(type="data", symmetric=False, array=part.q75-mid,
                     arrayminus=mid-part.q25, thickness=.8, width=2),
        customdata=part.n, hovertemplate="Day %{x} · %{y:.2f}<br>%{customdata} dates<extra></extra>"),
        row=row, col=2)
    fig_cycle.update_yaxes(title_text="VZA (°)" if row == 1 else label, nticks=3, row=row, col=1)
    fig_cycle.update_yaxes(rangemode="tozero", nticks=3, row=row, col=2)
    add_haiyan_marker(fig_cycle, row=row, col=1)

style_figure(fig_cycle, "", 1420, 1320)
fig_cycle.update_layout(showlegend=False, margin=dict(l=78, r=35, t=58, b=65))
fig_cycle.add_annotation(x=EVENT_DATE, y=1.005, xref="x", yref="paper", text="Haiyan",
    showarrow=False, xanchor="right", font=dict(color=EVENT_COLOR, size=13))
for x, anchor, text in [(0, "left", f"Daily G8 VZA · ◆ qualified {focus} dip"),
                        (1, "right", f"Pre-Haiyan · {cycle_view} profile")]:
    fig_cycle.add_annotation(x=x, y=1.04, xref="paper", yref="paper",
        xanchor=anchor, text=text, showarrow=False)
fig_cycle.update_xaxes(tickformat="%b %Y", dtick="M3", col=1)
fig_cycle.update_xaxes(tickvals=phase_ticks, col=2)
fig_cycle.update_xaxes(title_text=phase_label, row=9, col=2)

In [30]:
fig_cycle.show()
fig_cycle.write_image(FIGURE_DIR / f"05_ghsl_vza_phase_{cycle_view.replace( ' ', '_')}.png", scale=PNG_SCALE)

### Discussion

In the supplied 16-day profile, G1 tends to brighten around cycle days 3, 8 and 13, near peaks in regional VZA. Its phase medians rise from roughly 1.5 to 2.5–3 nW cm⁻² sr⁻¹. Broader masks show a similar alignment at lower radiance levels. These summaries group different dates and visible pixels. They establish an association with viewing phase while leaving its contribution to individual dips unresolved. The roughly monthly recurrence of near-zero events is a distinct result.

### Radiance by angle within pixels and months

In [31]:
vza_bands = {
    "<30°": valid_vza & (zenith_values < 30),
    "30–<50°": valid_vza & (zenith_values >= 30) & (zenith_values < 50),
    "≥50°": valid_vza & (zenith_values >= 50),
}
bins = {label: mask & pair for label, mask in vza_bands.items()}
log_ntl = np.log1p(paired_ntl)
month_index = daily_calendar.to_period("M")

fig_angles = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.12,
    subplot_titles=["HQ pixel-days", "Same pixel and month"],
)

support_rows = []
for label, angles in vza_bands.items():
    matched = bins[label]
    n_angles = int(angles.sum())
    n_matched = int(matched.sum())
    support_rows.append((label, n_angles, n_matched,
                         100 * n_matched / n_angles if n_angles else np.nan))

    if n_matched:
        p10, q1, mid, q3, p90 = np.quantile(log_ntl[matched], [.10, .25, .50, .75, .90])
        fig_angles.add_trace(go.Box(
            x=[label], q1=[q1], median=[mid], q3=[q3],
            lowerfence=[p10], upperfence=[p90],
            name=label, showlegend=False, marker_color=PRIMARY_COLOR,
        ), row=1, col=1)

vza_support = pd.DataFrame(support_rows, columns=[
    "VZA", "A1 pixel-days", "HQ matched pixel-days", "HQ matched (%)",
]).set_index("VZA")
display(vza_support.round(1))

contrasts = []
for month_id in month_index.unique():
    inside = month_index == month_id
    ref = bins["<30°"][inside]
    month_ntl = log_ntl[inside]
    ref_median = median(np.where(ref, month_ntl, np.nan), axis=0)

    for label in ("30–<50°", "≥50°"):
        other = bins[label][inside]
        eligible = (ref.sum(axis=0) >= 2) & (other.sum(axis=0) >= 2)
        idx = np.flatnonzero(eligible)
        if not idx.size:
            continue

        other_median = median(
            np.where(other[:, idx], month_ntl[:, idx], np.nan), axis=0
        )
        contrasts.append(pd.DataFrame({
            "month": month_id, "year": month_id.year, "angle_band": label,
            "pixel": g1_pixels[idx], "delta_log1p": other_median - ref_median[idx],
        }))

if contrasts:
    vza_contrasts = pd.concat(contrasts, ignore_index=True)
    per_pixel = (
        vza_contrasts.groupby(["year", "angle_band", "pixel"], as_index=False)
        .agg(delta_log1p=("delta_log1p", "median"), months=("month", "size"))
    )
    vza_summary = per_pixel.groupby(["year", "angle_band"]).agg(
        median=("delta_log1p", "median"), size=("pixel", "size"),
        pixel_months=("months", "sum"),
    )

    for year, color in [(2012, PRIMARY_COLOR), (2013, EXPANDED_COLOR)]:
        part = per_pixel.loc[per_pixel["year"].eq(year)]
        if not part.empty:
            fig_angles.add_trace(go.Box(
                x=part["angle_band"], y=part["delta_log1p"],
                name=str(year), marker_color=color, boxpoints=False,
            ), row=1, col=2)

    display(vza_summary.rename(columns={
        "median": "Median Δ log(1 + NTL)", "size": "Pixels", "pixel_months": "Pixel-months",
    }).round(3))
else:
    vza_summary = pd.DataFrame(columns=["year", "angle_band", "median", "size", "pixel_months"])
    fig_angles.add_annotation(
        text="Insufficient observations in both angle bands",
        x=.5, y=.5, xref="x2 domain", yref="y2 domain", showarrow=False,
    )

vza_summary.to_csv(TABLE_DIR / "pre_haiyan_vza_contrasts.csv", index=bool(contrasts))
fig_angles.add_hline(y=0, line_width=1, line_dash="dot", line_color="#8795A3", row=1, col=2)
style_figure(fig_angles, "", 1280, 495)
fig_angles.update_layout(boxmode="group", margin=dict(l=90, r=35, t=55, b=90))
fig_angles.update_yaxes(title_text="log(1 + NTL)", row=1, col=1)
fig_angles.update_yaxes(title_text="Δ log(1 + NTL) vs <30°", row=1, col=2)
fig_angles.update_xaxes(categoryorder="array", categoryarray=list(vza_bands), row=1, col=1)
fig_angles.update_xaxes(categoryorder="array", categoryarray=["30–<50°", "≥50°"], row=1, col=2)

,A1 pixel-days,HQ matched pixel-days,HQ matched (%)
VZA,,,
<30°,96704,27401,28.3
30–<50°,103090,34871,33.8
≥50°,128162,46633,36.4


Median Δ log(1 + NTL)  Pixels  Pixel-months
year angle_band                                             
2012 30–<50°                     0.064     503          3136
     ≥50°                        0.250     503          3387
2013 30–<50°                     0.079     503          2577
     ≥50°                        0.236     503          2937

In [32]:
fig_angles.show()
fig_angles.write_image(FIGURE_DIR / ("06_vza_angle_contrasts" + ".png"), scale=PNG_SCALE)

### Discussion

Pooled boxes weight pixels by their number of observations. The paired contrasts compare each pixel with its own low-angle observations in the same month, requiring two dates in each angle band. A positive contrast indicates brighter oblique observations. Interpret its magnitude alongside the number of pixels, pixel-months and HQ matches in each band.

## 7. Pre-Haiyan reference level and sensitivity

The primary level is the median of qualified G1 daily spatial medians in the final 60 days. P10–P90 describes observed daily variability. The sensitivity analysis retains both mean-based alternatives and a four-day median.

In [33]:
temporal_cap = primary["mean"].quantile(.95)
variants = {
    "G1 · daily mean": primary.mean_capped.where(primary.SC.ge(tau)),
    "G1 · temporal P95 mean": primary["mean"].clip(lower=0, upper=temporal_cap).where(primary.SC.ge(SPATIAL_COMPLETENESS_PCT)),
    "G1 · daily spatial median": primary["median"].where(primary.SC.ge(SPATIAL_COMPLETENESS_PCT)),
}
lookbacks = (30, BASELINE_DAYS, 90, 180)
reference_rows = []
reference_days = None
for days in sorted(set(lookbacks)):
    start = EVENT_DATE - pd.Timedelta(days=days)
    reference = {label: series.loc[start:PRE_EVENT_END] for label, series in variants.items()}
    daily_reference = reference["G1 · daily spatial median"]
    reference["G1 · W=4 (≥2 direct days)"] = daily_reference.rolling(
        AGGREGATION_DAYS, center=True, min_periods=2
    ).median()
    if days == BASELINE_DAYS:
        reference_days = reference
    for label, values in reference.items():
        level = values.median()
        quantiles = values.quantile([.10,.25,.75,.90])
        n_direct = daily_reference.count() if "W=4" in label else values.count()
        reference_rows.append((days, label, values.count(), n_direct, level,
                               *quantiles, (values-level).abs().median()))
baseline = pd.DataFrame(reference_rows, columns=[
    "Lookback days", "Configuration", "Estimates", "Direct days", "Level",
    "P10", "P25", "P75", "P90", "MAD",
])
baseline["Direct fraction (%)"] = 100 * baseline["Direct days"] / baseline["Lookback days"]
baseline["Meets minimum support"] = baseline["Direct days"].ge(MIN_REFERENCE_DAYS)
baseline.to_csv(TABLE_DIR / "pre_haiyan_baseline_summary.csv", index=False)
display(baseline.set_index(["Lookback days","Configuration"]).round(3))

Estimates  Direct days  Level    P10  \
Lookback days Configuration                                                     
30            G1 · daily mean                    9            9  3.292  2.616   
              G1 · temporal P95 mean             9            9  3.757  2.898   
              G1 · daily spatial median          9            9  2.162  1.075   
              G1 · W=4 (≥2 direct days)         12            9  2.294  2.074   
60            G1 · daily mean                   14           14  3.244  2.611   
              G1 · temporal P95 mean            14           14  3.665  2.880   
              G1 · daily spatial median         14           14  1.875  1.279   
              G1 · W=4 (≥2 direct days)         17           14  2.294  1.650   
90            G1 · daily mean                   23           23  3.668  2.610   
              G1 · temporal P95 mean            23           23  4.211  2.873   
              G1 · daily spatial median         23           23  1.786  1.212   
              G1 · W=4 (≥2 direct days)         28           23  2.139  1.604   
180           G1 · daily mean                   53           53  3.644  2.364   
              G1 · temporal P95 mean            53           53  4.057  2.646   
              G1 · daily spatial median         53           53  1.774  0.527   
              G1 · W=4 (≥2 direct days)         62           53  2.012  1.324   

                                           P25    P75    P90    MAD  \
Lookback days Configuration                                           
30            G1 · daily mean            2.747  4.218  4.393  0.684   
              G1 · temporal P95 mean     3.017  4.429  4.984  0.740   
              G1 · daily spatial median  1.612  2.613  2.882  0.456   
              G1 · W=4 (≥2 direct days)  2.150  2.351  2.562  0.152   
60            G1 · daily mean            2.747  4.183  4.738  0.631   
              G1 · temporal P95 mean     2.962  4.418  5.574  0.730   
              G1 · daily spatial median  1.598  2.538  3.541  0.435   
              G1 · W=4 (≥2 direct days)  2.115  2.573  3.880  0.225   
90            G1 · daily mean            2.779  4.236  6.160  0.858   
              G1 · temporal P95 mean     3.080  4.590  6.599  1.067   
              G1 · daily spatial median  1.583  2.463  3.883  0.441   
              G1 · W=4 (≥2 direct days)  1.709  2.351  3.298  0.378   
180           G1 · daily mean            2.747  4.218  5.270  0.889   
              G1 · temporal P95 mean     2.996  4.705  6.013  1.040   
              G1 · daily spatial median  1.365  2.306  3.471  0.532   
              G1 · W=4 (≥2 direct days)  1.650  2.259  2.510  0.300   

                                         Direct fraction (%)  \
Lookback days Configuration                                    
30            G1 · daily mean                         30.000   
              G1 · temporal P95 mean                  30.000   
              G1 · daily spatial median               30.000   
              G1 · W=4 (≥2 direct days)               30.000   
60            G1 · daily mean                         23.333   
              G1 · temporal P95 mean                  23.333   
              G1 · daily spatial median               23.333   
              G1 · W=4 (≥2 direct days)               23.333   
90            G1 · daily mean                         25.556   
              G1 · temporal P95 mean                  25.556   
              G1 · daily spatial median               25.556   
              G1 · W=4 (≥2 direct days)               25.556   
180           G1 · daily mean                         29.444   
              G1 · temporal P95 mean                  29.444   
              G1 · daily spatial median               29.444   
              G1 · W=4 (≥2 direct days)               29.444   

                                         Meets minimum support  
Lookback days Configuration                                     
30            G1 · d

In [34]:
fig_baseline = make_subplots(rows=2, cols=1, vertical_spacing=.15, row_heights=[.56,.44],
                    subplot_titles=[f"Final {BASELINE_DAYS} days · daily observations and four-day median",
                                    "Reference level across windows and configurations"])
final = baseline.loc[baseline["Lookback days"].eq(BASELINE_DAYS)].set_index("Configuration")
estimate = final.loc["G1 · daily spatial median"]
if estimate["Meets minimum support"]:
    fig_baseline.add_hrect(y0=estimate.P10, y1=estimate.P90, fillcolor=PRIMARY_COLOR,
                  opacity=.12, line_width=0, layer="below", row=1,col=1)
    fig_baseline.add_hline(y=estimate.Level, line_dash="dot", line_color=PRIMARY_COLOR, row=1,col=1)
for label, color, mode in [("G1 · daily spatial median", PRIMARY_COLOR, "markers"),
                           ("G1 · W=4 (≥2 direct days)", AGGREGATE_COLOR, "lines")]:
    values = reference_days[label]
    fig_baseline.add_trace(go.Scatter(x=values.index, y=values, name=label, mode=mode,
        marker=dict(color=color, size=7), line=dict(color=color), connectgaps=False), row=1,col=1)
for label, part in baseline.groupby("Configuration", sort=False):
    fig_baseline.add_trace(go.Scatter(x=part["Lookback days"], y=part.Level, name=label, mode="lines+markers",
        showlegend=label not in ("G1 · daily spatial median", "G1 · W=4 (≥2 direct days)"),
        line=dict(color={"G1 · daily spatial median": PRIMARY_COLOR,
                         "G1 · W=4 (≥2 direct days)": AGGREGATE_COLOR}.get(label)),
        marker=dict(symbol=np.where(part["Meets minimum support"], "circle", "circle-open")),
        customdata=part[["Direct days", "P10", "P90"]],
        hovertemplate="%{x} days<br>Level %{y:.3f}<br>%{customdata[0]} direct days"
                      "<br>P10–P90 %{customdata[1]:.3f}–%{customdata[2]:.3f}<extra></extra>"),
        row=2,col=1)
mark_reference(fig_baseline, (1,))
fig_baseline.update_xaxes(range=[BASELINE_START, EVENT_DATE+pd.Timedelta(days=1)], row=1,col=1)
fig_baseline.update_xaxes(title_text="Pre-event lookback (calendar days)", tickvals=sorted(set(lookbacks)), row=2,col=1)
fig_baseline.update_yaxes(title_text=NTL_UNITS)
style_figure(fig_baseline, "Pre-Haiyan reference level, variability and sensitivity", 1420, 900)

In [35]:
fig_baseline.show()
fig_baseline.write_image(FIGURE_DIR / ("07_pre_haiyan_baseline" + ".png"), scale=PNG_SCALE)

In [36]:
n_ref = int(estimate["Direct days"])
if estimate["Meets minimum support"]:
    print(f"Primary pre-Haiyan G1 reference: {estimate.Level:.3f} {NTL_UNITS}. "
          f"Empirical P10–P90: {estimate.P10:.3f}–{estimate.P90:.3f} {NTL_UNITS}. "
          f"Based on {n_ref}/{BASELINE_DAYS} daily observations at ≥{SPATIAL_COMPLETENESS_PCT:g}% SC.")
    same_rule = baseline.loc[baseline.Configuration.eq("G1 · daily spatial median") & baseline["Meets minimum support"], "Level"]
    print(f"Across supported lookbacks, the G1 daily reference ranges from {same_rule.min():.3f} "
          f"to {same_rule.max():.3f} {NTL_UNITS}.")
else:
    print(f"Final {BASELINE_DAYS}-day reference has {n_ref} qualified days. "
          f"At least {MIN_REFERENCE_DAYS} are required by the reporting criterion. "
          "Review the longer-window estimates and their direct support.")

Primary pre-Haiyan G1 reference: 1.875 nW cm⁻² sr⁻¹. Empirical P10–P90: 1.279–3.541 nW cm⁻² sr⁻¹. Based on 14/60 daily observations at ≥25% SC.
Across supported lookbacks, the G1 daily reference ranges from 1.774 to 1.875 nW cm⁻² sr⁻¹.


### Discussion

Read the final-window estimate with its direct-day count. Four-day windows can share observations and therefore produce more estimates than independent observation dates. Differences among supported lookbacks show how much the reference depends on window choice. The printed level and empirical range provide the quantities to carry into recovery analysis.

### Reference-window coverage across years

In [37]:
prior_start = BASELINE_START - pd.DateOffset(years=1)
prior_end = PRE_EVENT_END - pd.DateOffset(years=1)
windows = {
    "2012 · same calendar span": g1_qualified_median.loc[prior_start:prior_end],
    "2013 · final reference window": g1_qualified_median.loc[BASELINE_START:PRE_EVENT_END],
}

In [38]:
fig_reference = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[.72, .28],
                    subplot_titles=["Qualified G1 daily median", "Daily G1 coverage"], vertical_spacing=.12)
seasonal_sc, window_checks = {}, []
for (label, values), color in zip(windows.items(), [EXPANDED_COLOR, PRIMARY_COLOR]):
    offsets = pd.date_range(BASELINE_START, periods=len(values))
    seasonal_sc[label] = pd.Series(primary.SC.reindex(values.index).to_numpy(), index=offsets)
    valid = values.notna()
    runs = valid.ne(valid.shift()).cumsum()
    longest_gap = int((~valid).groupby(runs).sum().max()) if len(valid) else 0
    window_checks.append((label, len(values), values.count(), longest_gap, values.last_valid_index(),
                          values.median(), values.quantile(.10), values.quantile(.90)))
    fig_reference.add_trace(go.Scatter(x=offsets, y=values, name=label, mode="markers",
        marker=dict(color=color, size=7), text=values.index.strftime("%d %b %Y"),
        hovertemplate="%{text}<br>NTL %{y:.3f}<extra></extra>"), row=1,col=1)
sc_heatmap(fig_reference, pd.DataFrame(seasonal_sc), 2, bar_y=.12)
fig_reference.update_xaxes(tickformat="%d %b", title_text="Calendar date", row=2,col=1)
fig_reference.update_yaxes(title_text=NTL_UNITS, row=1,col=1)
style_figure(fig_reference, "Final reference window and same dates in 2012", 1390, 680)

In [39]:
fig_reference.show()
fig_reference.write_image(FIGURE_DIR / ("08_reference_calendar_support" + ".png"), scale=PNG_SCALE)

In [40]:
window_checks = pd.DataFrame(window_checks, columns=[
    "Period", "Calendar days", "Direct days", "Longest unqualified run (days)",
    "Last qualifying date", "Median", "P10", "P90",
]).set_index("Period")
display(window_checks.round({key: 3 for key in ("Median", "P10", "P90")}))
window_checks.to_csv(TABLE_DIR / "pre_haiyan_reference_calendar_support.csv")

,Calendar days,Direct days,Longest unqualified run (days),Last qualifying date,Median,P10,P90
Period,,,,,,,
2012 · same calendar span,60,14,16,2012-11-06,2.047,1.149,4.533
2013 · final reference window,60,14,13,2013-11-02,1.875,1.279,3.541


### Discussion

Calendar matching holds the season approximately fixed across years. The coverage strips, longest gap and last qualifying date show how well each window samples the approach to landfall. Similar medians can carry different uncertainty when one window contains a prolonged observation gap.

## Baseline interpretation

Use the supported 60-day G1 median, P10–P90, direct-day count and last observation date together. G1 rarely reached the near-zero threshold before Haiyan. Broader masks reached it repeatedly, including high-coverage dates. Their near-monthly recurrence and the 16-day alignment of typical brightness with viewing geometry describe different features of baseline variation.

For a post-event assessment, compare like settlement supports and inspect the viewing angles represented in the reference and event windows. The within-pixel contrasts inform whether an angle-conditioned baseline warrants testing. A sustained departure across qualified observations provides stronger evidence of change than a single low night. Observation gaps bound when shock and recovery can be resolved.